# Pay-For-API

## 개요

**Amazon Bedrock AgentCore Payments**를 사용하면 AI agent가 private key를 보유하거나
각 transaction마다 사람의 승인을 받지 않고도 digital service에 자동으로
결제할 수 있습니다.

이 사용 사례에서는 AgentCore Payments를 통해 유료 HTTP API의 metered access를 구매하는
Strands agent를 구축합니다. Seller는 AWS CDK를 통해 배포된 간단한 "Fun Facts" service로,
호출당 **$0.01**를 부과하는 AWS Lambda 함수가 지원하는 Amazon API Gateway HTTP API입니다.
Agent가 fact를 요청하면 seller는 machine-to-machine micropayment를 위해 예약된 402 status code를 활용하는 open spec인 [x402 protocol](https://x402.org)의 HTTP 402 response와 함께
payment requirement를 반환합니다. Agent는 해당 requirement를 AgentCore Payments의
`ProcessPayment` operation으로 전달하고 signed proof를 받은 후 proof를 첨부하여
request를 retry하고 유료 fact를 반환합니다. 이 모든 과정에서 agent는
private key를 전혀 다루지 않습니다.

내부적으로 AgentCore Payments가 wallet, signing key, on-chain settlement를
관리합니다. `PaymentManager`가 **Coinbase CDP** 또는 **Stripe via Privy**에
연결되어 있더라도 agent가 실행하는 코드는 동일하며 service가 instrument에 연결된
connector에서 적절한 signer를 선택합니다.

### 사용 사례 세부 정보

| 항목                | 세부 정보                                                             |
|:--------------------|:----------------------------------------------------------------------|
| 사용 사례 유형      | 자동 micropayment를 사용하는 agentic HTTP API 사용                    |
| AgentCore 구성 요소 | Amazon Bedrock AgentCore Payments                                     |
| Wallet provider     | Coinbase CDP ✅   ·   Stripe via Privy ✅                             |
| Payment protocol    | Wire의 x402(HTTP 402 Payment Required)                                |
| Agent 유형          | Single                                                                |
| Agentic Framework   | Strands Agents                                                        |
| LLM model           | Anthropic Claude Sonnet 4.5(Amazon Bedrock, `us.` inference profile)         |
| 난이도              | 중급                                                                  |
| 사용 SDK            | boto3                                                                 |

### 아키텍처

모든 유료 request에는 세 가지 role이 참여합니다.

1. **Strands agent** — `http_request` 호출, AgentCorePaymentsPlugin이 402 → ProcessPayment → retry 처리
2. **Amazon Bedrock AgentCore Payments** — `ProcessPayment`를 받고 instrument에 연결된
   wallet(Coinbase CDP 또는 Privy)을 사용하여 signed proof 반환
3. **Seller(CDK stack)** — 402 challenge를 발급하고 proof를 검증하여 content를 제공하는
   API Gateway 뒤의 Lambda

<div style="text-align:left">
    <img src="images/architecture_pay_for_api.png" alt="Pay-for-API architecture: a Strands buyer agent calls a Fun Facts seller through AgentCore Payments, which signs an x402 payment header and settles the USDC transfer on-chain via the configured wallet provider." width="75%"/>
</div>

**번호가 지정된 flow(diagram과 일치)**

1. **User**가 **Agent**(AgentCore Runtime + Strands)에 query를 보냅니다.
2. Agent가 **Amazon API Gateway** → **AWS Lambda**에서 hosting되는 유료 API를 호출합니다.
3. Seller가 **HTTP 402 Payment Required**와 payment requirement payload로 응답합니다.
4. Agent가 requirement를 **AgentCore Payments**로 전달하면 일치하는 `PaymentInstrument`를
   선택하고 session budget을 확인한 후 구성된 wallet provider(Coinbase CDP 또는
   Stripe via Privy)를 통해 payment에 sign합니다.
5. Agent가 signed `X-PAYMENT` header와 함께 request를 retry합니다. Seller가 x402 facilitator를
   통해 검증 및 on-chain settlement를 수행한 후 content와 함께 **200 OK**를 반환합니다.
6. Agent가 사용자에게 답하고 operator가 `GetPaymentSession`을 통해 지출을 감사합니다.

### 사용 사례 핵심 기능

* 설계상 agent는 private key를 보유하지 않습니다. AgentCore Payments가 구성된
  `PaymentManager` 및 `PaymentConnector`를 통해 모든 charge에 sign합니다.
* Wallet provider에 독립적 — 완전히 동일한 agent 코드가 Coinbase CDP instrument 또는
  Stripe-via-Privy instrument에서 실행됩니다.
* Payment session의 `maxSpendAmount`를 통해 사람이 budget 제어
* IAM role 분리: `ManagementRole`은 session을 생성하고 `ProcessPaymentRole`은 payment에
  sign합니다(양방향 명시적 `Deny`로 문서가 아니라 실제 적용).
* `GetPaymentSession`을 통한 전체 audit trail로 operator가 agent의 지출을
  확인합니다.

### API Reference

이 사용 사례에서 사용하는 AgentCore Payments API는 편의를 위해 repository에 포함된
service OpenAPI spec에 설명되어 있습니다.

| # | API | Plane | 용도 |
|---|-----|-------|---------|
| 1 | `CreatePaymentCredentialProvider` | Control | Wallet provider credentials를 안전하게 저장 |
| 2 | `CreatePaymentManager` | Control | Top-level payment processing resource |
| 3 | `CreatePaymentConnector` | Control | Credential Provider를 Manager에 연결 |
| 4 | `CreatePaymentInstrument` | Data | Manager 아래에 사용자의 embedded wallet provision |
| 5 | `GetPaymentInstrument` | Data | Service에서 할당한 vendor `userId` 조회(§7) |
| 6 | `CreatePaymentSession` | Data | Instrument로 scope가 제한된 budget limit session 생성 |
| 7 | `ProcessPayment`       | Data | 단일 x402 charge에 대한 cryptographic proof 생성 |
| 8 | `GetPaymentSession`    | Data | Session state + `availableLimits.availableSpendAmount` 조회 |
| 9 | `GetPaymentInstrumentBalance` | Data | Instrument wallet의 on-chain USDC balance |
| 10 | `ListPaymentInstruments` | Data | Manager 아래의 모든 instrument(user별 filter) |
| 11 | `ListPaymentSessions`  | Data | Manager 아래의 모든 session(user별 filter) |
| 12 | `DeletePaymentSession` | Data | Session hard-delete(revoke path) |
| 13 | `DeletePaymentInstrument` | Data | Instrument soft-delete(status가 `DELETED`로 변경) |

`bedrock-agentcore-control`(CP) 및 `bedrock-agentcore`(DP)의 boto3 service reference에서
전체 operation 목록과 response shape를
확인할 수 있습니다.

개념적 배경과 전체 AgentCore Payments API surface는
public documentation을 참조하세요.

- [AgentCore Payments overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments.html)
- [How it works](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-how-it-works.html)
- [Core concepts](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-concepts.html)
- [Prerequisites](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-prerequisites.html)
- [IAM roles](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-iam-roles.html)
- [Set up a credential provider](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-setup-credential-provider.html)
- [Create a payment manager](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-create-manager.html)
- [Create a payment instrument](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-create-instrument.html)
- [Create a payment session](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-create-session.html)
- [Process a payment](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-process-payment.html) — plugin reference, network preferences, interrupt contract
- [Connect to Bazaar](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-connect-bazaar.html)


## 사전 요구 사항

이 Notebook을 실행하려면 다음 항목이 필요합니다.

* 선택한 region에서 Amazon Bedrock AgentCore Payments를 사용할 수 있는 **AWS account**
* 선택한 region에서 **Anthropic Claude Sonnet 4.5** access가 활성화된 **Amazon Bedrock**
* **Python 3.10+** 및 Jupyter Notebook(또는 JupyterLab)
* Credentials로 구성된 **AWS CLI v2**(`aws configure`)
* 전역으로 설치된 **AWS CDK v2**(`npm install -g aws-cdk`) — seller 배포에 사용
* **Node.js 18+** — CDK에 필요
* Boto3 installation에서 사용할 수 있는 **AgentCore Payments botocore service definition**(boto3가 service 호출 방법을 인식하도록 함)
* **AgentCore Payments IAM role** — §2의 설정 셀에서 네 role(`ControlPlaneRole`, `ManagementRole`, `ProcessPaymentRole`, `ResourceRetrievalRole`) 생성
* **Wallet provider account** — Coinbase CDP(API Key ID, API Key Secret, Wallet Secret) 또는 Stripe via Privy(App ID, App Secret, Authorization Key ID, P-256 Authorization Private Key)
* **Base Sepolia** 및 **Solana Devnet** 모두에서 [Circle Faucet](https://faucet.circle.com/)의 **testnet USDC** — §5에서 network별 wallet을 하나씩 생성하고 provision 후 Notebook에서 입금


## 1. Dependency 설치

아래 셀을 실행하여 Notebook, seller, agent runtime에 필요한 모든 Python dependency를
설치합니다.


In [ ]:
!pip install -r requirements.txt --quiet

## 2. 환경 구성

Notebook은 이 folder의 `.env` 파일에서 모든 값을 읽습니다. 아래 셀을
실행하여 다음 작업을 수행합니다.

1. Notebook에서 assume할 네 IAM role 생성(idempotent하여
   안전하게 재실행 가능). `setup-roles.sh`가 role ARN을
   `.env`에 직접 작성합니다.
2. Editor tab에서 `.env`를 열고 아직 입력해야 할 값을 나열합니다.
   파일을 저장한 다음 계속하려면 **이 셀을 다시 실행**합니다.

다음 값이 필요합니다.

- `AWS_REGION` — one of `us-east-1`, `us-west-2`, `eu-central-1`,
  `ap-southeast-2`(AgentCore Payments preview region) 중 하나. Template에
  `us-west-2`로 설정되어 있으며 다른 region을 사용하려면 변경합니다.
- `COINBASE_API_KEY_ID`, `COINBASE_API_KEY_SECRET`, `COINBASE_WALLET_SECRET`
  — coinbase.com/developer-platform → Project → API Keys + Wallet에서 가져옵니다.
  사용하기 전에 Project → Wallet → Embedded Wallets → Policies에서
  *Delegated signing*을 활성화합니다.
- `PRIVY_APP_ID`, `PRIVY_APP_SECRET`, `PRIVY_AUTHORIZATION_ID`,
  `PRIVY_AUTHORIZATION_PRIVATE_KEY` — Privy dashboard → App →
  API Keys + Authorization Keys에서 가져옵니다. 붙여 넣기 전에 private key에서
  `wallet-auth:` prefix를 제거합니다.
- `INSTRUMENT_EMAIL` — 사용자가 관리하는 실제 inbox. Agent에 signing delegation을
  부여할 때 Coinbase Wallet Hub와 Privy가 모두 이 주소로 verification mail을
  보냅니다(§4.5).
- `SELLER_WALLET_ADDRESS`, `SELLER_SOLANA_WALLET_ADDRESS` — 사용자가 관리하는 임의의
  testnet address(seller는 자금만 수령).

이 셀을 처음 실행하면 `USER_ID`가 자동 생성됩니다.

---

#### Coinbase CDP — API + Wallet Credentials 수집

[Coinbase Developer Platform Portal](https://portal.cdp.coinbase.com/)을 열고 다음 작업을 수행합니다.

1. 로그인하거나 무료 account를 생성하고 **Project**를 선택 또는 생성합니다.
2. **API Keys** → **Create API key**로 이동하여 다음 값을 복사합니다.
   - **Key ID** → `COINBASE_API_KEY_ID`
   - **Key Secret** → `COINBASE_API_KEY_SECRET`
3. **Wallet** → **Wallet Secret** → **Generate**로 이동하여 값을
   `COINBASE_WALLET_SECRET`에 복사합니다.
4. **Wallet** → **Embedded Wallets** → **Policies**로 이동하여
   **Delegated signing**을 활성화합니다. 이 설정이 없으면 AgentCore Payments가
   사용자를 대신해 sign할 수 없습니다.

#### Stripe via Privy — App + Authorization Key 수집

[Privy Dashboard](https://dashboard.privy.io/)를 열고 다음 작업을 수행합니다.

1. 로그인하고 **App**을 선택 또는 생성합니다.
2. **App settings** → **API Keys**로 이동하여 다음 값을 복사합니다.
   - **App ID** → `PRIVY_APP_ID`
   - **App secret** → `PRIVY_APP_SECRET`
3. **App settings** → **Authorization Keys** → **Create new key**로 이동합니다.
   다음 값을 복사합니다.
   - **Key ID** → `PRIVY_AUTHORIZATION_ID`
   - **Private key** → `PRIVY_AUTHORIZATION_PRIVATE_KEY`. Dashboard에서 값 앞에
     `wallet-auth:` prefix를 추가하므로 붙여 넣기 전에 prefix를
     제거합니다.


In [ ]:
# 1단계: IAM role 생성. setup-roles.sh가 네 role의
# ARN을 .env에 직접 작성(idempotent하여 안전하게 재실행 가능)
import shutil
import subprocess
from pathlib import Path

USE_CASE = Path(".").resolve()
ENV_FILE = USE_CASE / ".env"
TEMPLATE = USE_CASE / "env-sample.txt"

if not ENV_FILE.exists():
    shutil.copy2(TEMPLATE, ENV_FILE)
    print(f"✅ Seeded {ENV_FILE.name} from {TEMPLATE.name}")

roles_proc = subprocess.run(
    ["bash", "test/integration/setup-roles.sh"],
    check=False,
)
if roles_proc.returncode != 0:
    raise RuntimeError(
        f"setup-roles.sh exited {roles_proc.returncode} — fix the error from the previous step and re-run this cell."
    )

# 2단계: .env에 수동으로 입력해야 하는 key 안내
# 아직 비어 있는 항목을 확인하여 정확히 나열함. 사용자가 열린 .env tab에
# 값을 붙여 넣고(`code` CLI를 사용할 수 없으면 직접 열기) 저장한 다음
# 이 셀을 다시 실행
REQUIRED_MANUAL = [
    ("COINBASE_API_KEY_ID", "Coinbase CDP API key ID (coinbase.com/developer-platform → Project → API Keys)"),
    ("COINBASE_API_KEY_SECRET", "Coinbase CDP API key secret"),
    ("COINBASE_WALLET_SECRET", "Coinbase CDP wallet secret (Project → Wallet; enable Delegated signing first)"),
    ("PRIVY_APP_ID", "Privy App ID (Privy dashboard → App → API Keys)"),
    ("PRIVY_APP_SECRET", "Privy App Secret"),
    ("PRIVY_AUTHORIZATION_ID", "Privy Authorization Key ID"),
    ("PRIVY_AUTHORIZATION_PRIVATE_KEY", "Privy P-256 Authorization Private Key (strip 'wallet-auth:' prefix)"),
    ("INSTRUMENT_EMAIL", "Real inbox you control — Coinbase Wallet Hub and Privy send verification here"),
    ("SELLER_WALLET_ADDRESS", "EVM payout address (Base Sepolia) — any testnet 0x… address you control"),
    ("SELLER_SOLANA_WALLET_ADDRESS", "Solana payout address (Solana Devnet) — any base58 address you control"),
]

# 비어 있는 항목을 확인하도록 현재 .env를 dict로 parsing
env_values = {}
for line in ENV_FILE.read_text().splitlines():
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1)
        env_values[k.strip()] = v.strip()

missing = [(key, hint) for key, hint in REQUIRED_MANUAL if not env_values.get(key) or env_values[key].startswith("<")]

# 첫 실행 시 USER_ID를 UUID로 자동 생성하여 Notebook 실행 간
# session 충돌 방지. 현재 비어 있을 때만 작성
if not env_values.get("USER_ID"):
    import uuid
    from utils import write_env_updates

    write_env_updates({"USER_ID": f"pay-for-api-{uuid.uuid4()}"}, env_path=str(ENV_FILE))
    print("✅ Auto-wrote USER_ID as a fresh UUID")

if missing:
    print(f"\n⏳ Fill in these {len(missing)} values in {ENV_FILE.name}:\n")
    for key, hint in missing:
        print(f"   • {key}")
        print(f"       {hint}")
    print(
        f"\n   Opening {ENV_FILE.name} in an editor tab. Paste the values,\n"
        "   save, then re-run the NEXT cell (Environment check)."
    )
    # VS Code는 PATH에 `code`를 제공하며 사용할 수 없으면 수동 안내로 fallback
    try:
        subprocess.run(["code", str(ENV_FILE)], check=False)
    except FileNotFoundError:
        print(f"\n   Open manually: {ENV_FILE}")
else:
    print("\n✅ All required .env values are set. Move on to the next cell.")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# ── AWS 구성 ────────────────────────────────────────────────────────────────
# Boto3가 AWS_REGION에서 AgentCore Payments control-plane 및 data-plane
# endpoint를 resolve하므로 여기서는 endpoint URL 불필요
AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")

# ── IAM role ARN(setup-roles.sh에서 가져옴) ─────────────────────────────────
CONTROL_PLANE_ROLE_ARN = os.environ.get("CONTROL_PLANE_ROLE_ARN", "")
MANAGEMENT_ROLE_ARN = os.environ.get("MANAGEMENT_ROLE_ARN", "")
PROCESS_PAYMENT_ROLE_ARN = os.environ.get("PROCESS_PAYMENT_ROLE_ARN", "")
RESOURCE_RETRIEVAL_ROLE_ARN = os.environ.get("RESOURCE_RETRIEVAL_ROLE_ARN", "")

# ── Wallet provider secret 설정 ─────────────────────────────────────────────
# Notebook이 동일한 Manager 아래에 두 provider를 함께 연결함.
# .env에 두 credential set을 모두 제공. 하나를 건너뛰려면 해당 set을
# 비워 둘 수 있으며 설정 셀에서 확인 후 경고
COINBASE_API_KEY_ID = os.environ.get("COINBASE_API_KEY_ID", "")
COINBASE_API_KEY_SECRET = os.environ.get("COINBASE_API_KEY_SECRET", "")
COINBASE_WALLET_SECRET = os.environ.get("COINBASE_WALLET_SECRET", "")

PRIVY_APP_ID = os.environ.get("PRIVY_APP_ID", "")
PRIVY_APP_SECRET = os.environ.get("PRIVY_APP_SECRET", "")
PRIVY_AUTHORIZATION_ID = os.environ.get("PRIVY_AUTHORIZATION_ID", "")
PRIVY_AUTHORIZATION_PRIVATE_KEY = os.environ.get("PRIVY_AUTHORIZATION_PRIVATE_KEY", "")

# ── §4 / §5에서 채우는 state ─────────────────────────────────────────────
# Manager 하나, Connector 두 개(provider별 하나), Instrument 네 개
# (provider별 EVM + SOL), Session 두 개(provider별 하나)
MANAGER_ARN = os.environ.get("MANAGER_ARN", "")
MANAGER_ID = os.environ.get("MANAGER_ID", "")

CREDENTIAL_PROVIDER_ARN_CDP = os.environ.get("CREDENTIAL_PROVIDER_ARN_CDP", "")
CREDENTIAL_PROVIDER_ARN_PRIVY = os.environ.get("CREDENTIAL_PROVIDER_ARN_PRIVY", "")

CONNECTOR_ID_CDP = os.environ.get("CONNECTOR_ID_CDP", "")
CONNECTOR_ID_PRIVY = os.environ.get("CONNECTOR_ID_PRIVY", "")

PAYMENT_INSTRUMENT_ID_CDP_EVM = os.environ.get("PAYMENT_INSTRUMENT_ID_CDP_EVM", "")
WALLET_ADDRESS_CDP_EVM = os.environ.get("WALLET_ADDRESS_CDP_EVM", "")
PAYMENT_INSTRUMENT_ID_CDP_SOL = os.environ.get("PAYMENT_INSTRUMENT_ID_CDP_SOL", "")
WALLET_ADDRESS_CDP_SOL = os.environ.get("WALLET_ADDRESS_CDP_SOL", "")
PAYMENT_INSTRUMENT_ID_PRIVY_EVM = os.environ.get("PAYMENT_INSTRUMENT_ID_PRIVY_EVM", "")
WALLET_ADDRESS_PRIVY_EVM = os.environ.get("WALLET_ADDRESS_PRIVY_EVM", "")
PAYMENT_INSTRUMENT_ID_PRIVY_SOL = os.environ.get("PAYMENT_INSTRUMENT_ID_PRIVY_SOL", "")
WALLET_ADDRESS_PRIVY_SOL = os.environ.get("WALLET_ADDRESS_PRIVY_SOL", "")

SESSION_ID_CDP = os.environ.get("SESSION_ID_CDP", "")
SESSION_ID_PRIVY = os.environ.get("SESSION_ID_PRIVY", "")

SELLER_API_URL = os.environ.get("SELLER_API_URL", "").rstrip("/")

# Seller payout wallet — seller 배포 시(§3) 사용하고 여기서 확인하여
# 배포 전 오타를 발견하도록 함. Agent는 읽지 않음
SELLER_WALLET_ADDRESS = os.environ.get("SELLER_WALLET_ADDRESS", "")
SELLER_SOLANA_WALLET_ADDRESS = os.environ.get("SELLER_SOLANA_WALLET_ADDRESS", "")

# ── Session 및 user config ─────────────────────────────────────────────────
# USER_ID는 §2에서 자동 생성됨. 설정되지 않은 상태로 여기에 도달하면
# CreatePaymentInstrument의 필수 header가 채워지도록 UUID로 fallback
# (service는 빈 문자열 거부)
import uuid as _uuid

USER_ID = os.environ.get("USER_ID") or f"pay-for-api-{_uuid.uuid4()}"
INSTRUMENT_EMAIL = os.environ.get("INSTRUMENT_EMAIL", f"{USER_ID}@example.com")
SESSION_MAX_SPEND = os.environ.get("SESSION_MAX_SPEND", "1.00")
SESSION_EXPIRY_MINUTES = int(os.environ.get("SESSION_EXPIRY_MINUTES", "30"))


def _check(label: str, value: str, redact: bool = False, optional: bool = False) -> bool:
    ok = bool(value) and not value.startswith("<")
    display = "[redacted]" if redact and value else value
    if optional and not ok:
        display = display or "(will be set later)"
        print(f"  ⏳  {label}: {display}")
        return True
    icon = "✅" if ok else "❌ MISSING"
    print(f"  {icon}  {label}: {display}")
    return ok


_required_results = []


def _req(label: str, value: str, redact: bool = False) -> None:
    _required_results.append(_check(label, value, redact=redact, optional=False))


print("=== Environment check ===")
print("\n  AWS:")
_req("AWS_REGION", AWS_REGION)

print("\n  IAM roles (from setup-roles.sh):")
_req("CONTROL_PLANE_ROLE_ARN", CONTROL_PLANE_ROLE_ARN)
_req("MANAGEMENT_ROLE_ARN", MANAGEMENT_ROLE_ARN)
_req("PROCESS_PAYMENT_ROLE_ARN", PROCESS_PAYMENT_ROLE_ARN)
_req("RESOURCE_RETRIEVAL_ROLE_ARN", RESOURCE_RETRIEVAL_ROLE_ARN)

print("\n  Coinbase CDP secrets:")
_req("COINBASE_API_KEY_ID", COINBASE_API_KEY_ID)
_req("COINBASE_API_KEY_SECRET", COINBASE_API_KEY_SECRET, redact=True)
_req("COINBASE_WALLET_SECRET", COINBASE_WALLET_SECRET, redact=True)

print("\n  Stripe-via-Privy secrets:")
_req("PRIVY_APP_ID", PRIVY_APP_ID)
_req("PRIVY_APP_SECRET", PRIVY_APP_SECRET, redact=True)
_req("PRIVY_AUTHORIZATION_ID", PRIVY_AUTHORIZATION_ID)
_req("PRIVY_AUTHORIZATION_PRIVATE_KEY", PRIVY_AUTHORIZATION_PRIVATE_KEY, redact=True)

print("\n  Seller payout wallets (used by §3 deploy):")
_req("SELLER_WALLET_ADDRESS", SELLER_WALLET_ADDRESS)
_req("SELLER_SOLANA_WALLET_ADDRESS", SELLER_SOLANA_WALLET_ADDRESS)

print("\n  Wallet linked email (used by §4.5 CreatePaymentInstrument):")
_req("INSTRUMENT_EMAIL", INSTRUMENT_EMAIL)

print("\n  Populated later — ⏳ is expected on first run:")
print("  (§3 writes SELLER_API_URL; §4/§5 write Manager, Connector, Instrument, Session IDs)")
_check("MANAGER_ARN", MANAGER_ARN, optional=True)
_check("PAYMENT_INSTRUMENT_ID_CDP_EVM", PAYMENT_INSTRUMENT_ID_CDP_EVM, optional=True)
_check("PAYMENT_INSTRUMENT_ID_CDP_SOL", PAYMENT_INSTRUMENT_ID_CDP_SOL, optional=True)
_check("PAYMENT_INSTRUMENT_ID_PRIVY_EVM", PAYMENT_INSTRUMENT_ID_PRIVY_EVM, optional=True)
_check("PAYMENT_INSTRUMENT_ID_PRIVY_SOL", PAYMENT_INSTRUMENT_ID_PRIVY_SOL, optional=True)
_check("SELLER_API_URL", SELLER_API_URL, optional=True)

# Guard: 필수 값이 누락되면 여기서 중단하여 사용자가 .env 입력을 마치기 전에
# Run All이 §3 이후로 계속 실행되지 않도록 함
if not all(_required_results):
    missing_count = _required_results.count(False)
    raise SystemExit(
        f"{missing_count} required value(s) missing from the previous step. Fill them in "
        f".env (see the list printed by §2), save, then re-run this cell."
    )

## 3. Fun Facts Seller 배포

Seller는 AWS CDK stack으로 package된 AWS Lambda 기반 Amazon API Gateway HTTP API입니다.
두 endpoint를 제공합니다.

* `GET /health` — public이며 API metadata(price, network) 반환
* `GET /facts?topic=<topic>` — 유료이며 유효한 `X-PAYMENT` header가 첨부되지 않으면
  HTTP 402 반환

아래 셀은 이 account/region에서 필요할 경우 CDK를 bootstrap하고, `cdk deploy`를 실행한 후
생성된 API Gateway URL을 `SELLER_API_URL`로 `.env`에 다시 작성하는
`deploy-seller.sh`를 호출합니다.

> 💡 **팁:** 이 셀을 실행하기 전에 seller에서 settlement된 자금을 받을 address를 지정하도록
> `PAY_TO` environment variable을 설정하세요. 설정하지 않으면 CDK가 placeholder로
> 배포되고 facilitator가 verification 시
> proof를 거부합니다.

> ⚠️ **비용 안내:** Amazon API Gateway HTTP API와 AWS Lambda 함수를
> 배포합니다. 둘 다 request별 요금이 부과되며 튜토리얼 사용량은 일반적으로
> AWS Free Tier 내에 충분히 포함됩니다. 작업을 마치면 §10을 실행하여
> 제거하세요.

In [ ]:
import pathlib
import subprocess

HERE = pathlib.Path(".").resolve()
DEPLOY = HERE / "test" / "integration" / "deploy-seller.sh"

if not DEPLOY.exists():
    raise FileNotFoundError(f"Expected {DEPLOY} — run this notebook from the 02-use-cases/01-pay-for-api directory.")

# 긴 CDK 배포가 중단된 것처럼 보이지 않도록 output streaming
proc = subprocess.Popen(
    ["bash", str(DEPLOY)],
    cwd=str(HERE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"deploy-seller.sh failed with exit code {rc}")

# Script가 outputs.json에 작성한 URL 다시 load
import json as _json

OUTPUTS_FILE = HERE / "seller" / "cdk" / "outputs.json"
if OUTPUTS_FILE.exists():
    outs = _json.loads(OUTPUTS_FILE.read_text())
    SELLER_API_URL = outs["AgentCorePaymentsFunFactsSellerStack"]["SellerApiUrl"].rstrip("/")
    print(f"\n✅ SELLER_API_URL: {SELLER_API_URL}")
else:
    print("\n⚠️  Could not read seller/cdk/outputs.json — paste the URL into .env manually.")

### Seller 정상 작동 확인

`/health`를 호출하여 seller가 실행 중인지 확인합니다. 이 endpoint에는 payment가 필요하지 않습니다.


In [ ]:
import requests

health = requests.get(f"{SELLER_API_URL}/health", timeout=10)
health.raise_for_status()
print("Seller health:", health.json())

### 402 Response 미리 보기

Agent가 첫 호출에서 보게 될 정확한 내용입니다. `accepts[0]` entry가 x402 payment
requirement이며 agent는 §6에서 이를 `ProcessPayment`에 전달합니다.


In [ ]:
from utils import pp

preview = requests.get(f"{SELLER_API_URL}/facts?topic=space", timeout=10)
print(f"Status: {preview.status_code}")

# Seller는 payment requirement를 설명하는 JSON body와 함께 402 반환
# 일부 x402 middleware version은 Accept header가 application/json일 때 `{}`를,
# text일 때 빈 body를 반환하므로
# 두 경우 모두 처리
try:
    body = preview.json()
except ValueError:
    body = {}
pp("402 response body", body)

# Multi-network accepts: 구성된 payout wallet별 entry 하나
# 각 entry에는 `scheme`, `price`(사람이 읽을 수 있는 USD 문자열),
# `network`(Chain Agnostic Improvement Proposal 2(CAIP-2) 식별자, 예: `eip155:84532`), `payTo`가 있음. x402
# middleware가 `$0.01`을 on-chain atomic amount(token의 나눌 수 없는 최소 단위, USDC는 0.000001 USDC)로 변환
accepts = body.get("accepts") or []
if not accepts:
    print(
        "ℹ️  No `accepts[]` array in the 402 body.\n"
        "   This is expected for plain browser GETs against some x402\n"
        "   facilitator versions — the agent sends an `Accept-Payment`\n"
        "   header in §7 which triggers the full payment requirement\n"
        "   payload. You can skip this preview and continue."
    )
else:
    for i, entry in enumerate(accepts):
        print(
            f"  [{i}] scheme: {entry['scheme']} | "
            f"network: {entry['network']} | "
            f"price: {entry['price']} | "
            f"payTo: {entry['payTo'] or '(unset)'}"
        )

## 4. AgentCore Payments 설정

이 섹션에서는 AgentCore Payments에 필요한 모든 항목을 provision합니다.

1. **Credential Provider 2개** — Coinbase CDP용 하나, Stripe via Privy용 하나
2. **Payment Manager 1개** — top-level payment resource
3. **Payment Connector 2개** — provider별 하나, 동일한 Manager에 연결
4. **Payment Instrument 4개** — provider별 EVM + SOLANA. Agent는 §7에서
   각 (provider, network) pair마다 한 번 실행됩니다.

Kernel을 다시 시작해도 중단한 지점부터 계속할 수 있도록 각 단계의 출력을
`.env`에 유지합니다.

### 4.1 Role Assume 및 Client 구축


In [ ]:
import boto3
from boto3.session import Session
from utils import assume_role

missing_roles = [
    name
    for name, value in (
        ("CONTROL_PLANE_ROLE_ARN", CONTROL_PLANE_ROLE_ARN),
        ("MANAGEMENT_ROLE_ARN", MANAGEMENT_ROLE_ARN),
        ("RESOURCE_RETRIEVAL_ROLE_ARN", RESOURCE_RETRIEVAL_ROLE_ARN),
    )
    if not value
]
if missing_roles:
    raise RuntimeError(f"Missing IAM roles: {missing_roles}. Run `bash test/integration/setup-roles.sh`.")

boto_session = Session(region_name=AWS_REGION)

print("Assuming ControlPlaneRole...")
cp_session = assume_role(
    boto_session,
    CONTROL_PLANE_ROLE_ARN,
    session_name="pay-for-api-cp",
)
cp_client = cp_session.client("bedrock-agentcore-control")
cred_client = cp_session.client("bedrock-agentcore-control")

print("\nAssuming ManagementRole...")
mgmt_session = assume_role(
    boto_session,
    MANAGEMENT_ROLE_ARN,
    session_name="pay-for-api-mgmt",
)
dp_client_mgmt = mgmt_session.client("bedrock-agentcore")

# INSTRUMENT_EMAIL은 §2에서 .env로부터 이미 load됨. CreatePaymentInstrument는
# 모든 wallet에 이를 요구하며 사용자가 signing delegation을 부여할 때 Coinbase와 Privy 모두
# 해당 address로 verification mail 전송
print(f"\n✅ Clients ready")
print(f"   INSTRUMENT_EMAIL: {INSTRUMENT_EMAIL}")

### 4.2 Credential Provider 생성

`PaymentCredentialProvider`는 이 사용 사례의 security handoff입니다.
Notebook은 `.env`에서 wallet provider secret을 한 번 읽고 AgentCore Identity에
`CreatePaymentCredentialProvider`를 호출합니다.
Service는 API key, app secret, wallet 또는 authorization secret을
**AWS KMS** encryption이 적용된 **AWS Secrets Manager**에 저장하고
agent에는 secret ARN만 노출합니다. 이 셀을 실행한 후 secret material은
AgentCore-managed vault에 저장됩니다.
Agent runtime은 signing 시 `GetResourcePaymentToken`을 통해 단기 vendor-specific token을
가져오며 raw secret을 전혀
받지 않습니다. 로컬 `.env` copy는 runtime에 더 이상 필요하지 않으며
disk에서 제거하려면 수동으로 지울 수 있습니다.

나중에 둘 다 동일한 Manager에 연결할 수 있도록 Notebook은
**vendor별 provider 하나**를 생성합니다.

자세한 내용은 [Payment credential provider 생성](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/resource-providers.html#creating-a-payment-credential-provider)을 참조하세요.

#### 4.2a Coinbase CDP


In [ ]:
import uuid
from utils import idempotent_create, write_env_updates


def _client_token() -> str:
    # AgentCore Payments는 33자 이상의 `clientToken` 요구(idempotency token으로 retry 시 동일한 문자열을 전달하면 중복 생성 대신 기존 resource 반환). UUID v4(36자)는 안전함
    return str(uuid.uuid4()) + "-" + str(uuid.uuid4())[:8]


if CREDENTIAL_PROVIDER_ARN_CDP:
    print(f"↷ CREDENTIAL_PROVIDER_ARN_CDP already set in .env, skipping Coinbase CDP credential provider create")
else:
    missing_cdp = [
        n
        for n, v in (
            ("COINBASE_API_KEY_ID", COINBASE_API_KEY_ID),
            ("COINBASE_API_KEY_SECRET", COINBASE_API_KEY_SECRET),
            ("COINBASE_WALLET_SECRET", COINBASE_WALLET_SECRET),
        )
        if not v
    ]
    if missing_cdp:
        raise RuntimeError(f"Missing Coinbase CDP secrets in .env: {missing_cdp}")

    CRED_PROVIDER_NAME_CDP = f"PayForApiCDP{uuid.uuid4().hex[:8]}"
    resp = idempotent_create(
        cred_client.create_payment_credential_provider,
        conflict_msg=f"Credential provider {CRED_PROVIDER_NAME_CDP} already exists",
        name=CRED_PROVIDER_NAME_CDP,
        credentialProviderVendor="CoinbaseCDP",
        providerConfigurationInput={
            "coinbaseCdpConfiguration": {
                "apiKeyId": COINBASE_API_KEY_ID,
                "apiKeySecret": COINBASE_API_KEY_SECRET,
                "walletSecret": COINBASE_WALLET_SECRET,
            }
        },
    )
    if resp is None:
        raise RuntimeError("Credential provider already exists — rename or delete it.")
    CREDENTIAL_PROVIDER_ARN_CDP = resp["credentialProviderArn"]
    print(f"✅ Coinbase CDP credential provider: {CREDENTIAL_PROVIDER_ARN_CDP}")

    write_env_updates(
        {
            "CRED_PROVIDER_NAME_CDP": CRED_PROVIDER_NAME_CDP,
            "CREDENTIAL_PROVIDER_ARN_CDP": CREDENTIAL_PROVIDER_ARN_CDP,
        }
    )
    print("💾 .env updated: CRED_PROVIDER_NAME_CDP, CREDENTIAL_PROVIDER_ARN_CDP")

#### 4.2b Stripe via Privy


In [ ]:
if CREDENTIAL_PROVIDER_ARN_PRIVY:
    print(f"↷ CREDENTIAL_PROVIDER_ARN_PRIVY already set in .env, skipping Privy credential provider create")
else:
    missing_privy = [
        n
        for n, v in (
            ("PRIVY_APP_ID", PRIVY_APP_ID),
            ("PRIVY_APP_SECRET", PRIVY_APP_SECRET),
            ("PRIVY_AUTHORIZATION_ID", PRIVY_AUTHORIZATION_ID),
            ("PRIVY_AUTHORIZATION_PRIVATE_KEY", PRIVY_AUTHORIZATION_PRIVATE_KEY),
        )
        if not v
    ]
    if missing_privy:
        raise RuntimeError(f"Missing Privy secrets in .env: {missing_privy}")

    CRED_PROVIDER_NAME_PRIVY = f"PayForApiPrivy{uuid.uuid4().hex[:8]}"
    resp = idempotent_create(
        cred_client.create_payment_credential_provider,
        conflict_msg=f"Credential provider {CRED_PROVIDER_NAME_PRIVY} already exists",
        name=CRED_PROVIDER_NAME_PRIVY,
        credentialProviderVendor="StripePrivy",
        providerConfigurationInput={
            "stripePrivyConfiguration": {
                "appId": PRIVY_APP_ID,
                "appSecret": PRIVY_APP_SECRET,
                "authorizationId": PRIVY_AUTHORIZATION_ID,
                "authorizationPrivateKey": PRIVY_AUTHORIZATION_PRIVATE_KEY,
            }
        },
    )
    if resp is None:
        raise RuntimeError("Credential provider already exists — rename or delete it.")
    CREDENTIAL_PROVIDER_ARN_PRIVY = resp["credentialProviderArn"]
    print(f"✅ Stripe Privy credential provider: {CREDENTIAL_PROVIDER_ARN_PRIVY}")

    write_env_updates(
        {
            "CRED_PROVIDER_NAME_PRIVY": CRED_PROVIDER_NAME_PRIVY,
            "CREDENTIAL_PROVIDER_ARN_PRIVY": CREDENTIAL_PROVIDER_ARN_PRIVY,
        }
    )
    print("💾 .env updated: CRED_PROVIDER_NAME_PRIVY, CREDENTIAL_PROVIDER_ARN_PRIVY")

### 4.3 Payment Manager 생성

`PaymentManager`는 top-level payment resource입니다. Runtime에 위에서 저장한
credentials를 가져오기 위해 AgentCore Payments가 assume하는 role인
`RESOURCE_RETRIEVAL_ROLE_ARN`의 ARN을 받습니다.

Manager 생성은 async 방식이므로 `status == READY`가 될 때까지
`GetPaymentManager`를 polling합니다.


In [ ]:
import re
from utils import wait_for_status

if MANAGER_ARN:
    print(f"↷ MANAGER_ARN already set in .env, skipping create:\n   {MANAGER_ARN}")
else:
    # PaymentManager name: 영문자 + 숫자만 허용, 최대 48자, hyphen/underscore 불가
    MANAGER_NAME = f"PayForApi{uuid.uuid4().hex[:8]}"
    assert re.match(r"^[a-zA-Z][a-zA-Z0-9]{0,47}$", MANAGER_NAME), MANAGER_NAME

    resp = cp_client.create_payment_manager(
        name=MANAGER_NAME,
        description=f"AgentCore Payments Pay for API use case {MANAGER_NAME}",
        authorizerType="AWS_IAM",
        roleArn=RESOURCE_RETRIEVAL_ROLE_ARN,
        clientToken=_client_token(),
    )
    MANAGER_ID = resp["paymentManagerId"]
    MANAGER_ARN = resp["paymentManagerArn"]
    print(f"✅ Payment Manager created")
    print(f"   Manager ID:  {MANAGER_ID}")
    print(f"   Manager ARN: {MANAGER_ARN}")

    print("\nWaiting for PaymentManager to reach READY...")
    wait_for_status(
        cp_client.get_payment_manager,
        expected_status="READY",
        poll_interval=5,
        timeout=120,
        paymentManagerId=MANAGER_ID,
    )
    print("✅ PaymentManager is READY")

    write_env_updates(
        {
            "MANAGER_ID": MANAGER_ID,
            "MANAGER_ARN": MANAGER_ARN,
        }
    )
    print("💾 .env updated: MANAGER_ID, MANAGER_ARN")

### 4.4 Payment Connector 생성

`PaymentConnector`는 Credential Provider를 Manager에 연결합니다. Connector의 `type`은
AgentCore Payments에 사용할 signer를 알려 줍니다. 동일한 Manager 아래에 provider별로
하나씩 총 **두 개**의 connector를 생성합니다.

#### 4.4a Coinbase CDP Connector


In [ ]:
if CONNECTOR_ID_CDP:
    print(f"↷ CONNECTOR_ID_CDP already set in .env, skipping Coinbase CDP connector create")
else:
    CONNECTOR_NAME_CDP = f"PayForApiCDPConn{uuid.uuid4().hex[:6]}"
    resp = cp_client.create_payment_connector(
        paymentManagerId=MANAGER_ID,
        name=CONNECTOR_NAME_CDP,
        description=f"AgentCore Payments CoinbaseCDP connector {CONNECTOR_NAME_CDP}",
        type="CoinbaseCDP",
        credentialProviderConfigurations=[{"coinbaseCDP": {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN_CDP}}],
        clientToken=_client_token(),
    )
    CONNECTOR_ID_CDP = resp["paymentConnectorId"]
    print(f"✅ Coinbase CDP connector: {CONNECTOR_ID_CDP}")

    print("\nWaiting for CDP connector to reach READY...")
    wait_for_status(
        cp_client.get_payment_connector,
        expected_status="READY",
        poll_interval=5,
        timeout=120,
        paymentManagerId=MANAGER_ID,
        paymentConnectorId=CONNECTOR_ID_CDP,
    )
    print("✅ CDP connector is READY")

    write_env_updates({"CONNECTOR_ID_CDP": CONNECTOR_ID_CDP})
    print("💾 .env updated: CONNECTOR_ID_CDP")

#### 4.4b Stripe via Privy Connector


In [ ]:
if CONNECTOR_ID_PRIVY:
    print(f"↷ CONNECTOR_ID_PRIVY already set in .env, skipping Privy connector create")
else:
    CONNECTOR_NAME_PRIVY = f"PayForApiPrivyConn{uuid.uuid4().hex[:6]}"
    resp = cp_client.create_payment_connector(
        paymentManagerId=MANAGER_ID,
        name=CONNECTOR_NAME_PRIVY,
        description=f"AgentCore Payments StripePrivy connector {CONNECTOR_NAME_PRIVY}",
        type="StripePrivy",
        credentialProviderConfigurations=[{"stripePrivy": {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN_PRIVY}}],
        clientToken=_client_token(),
    )
    CONNECTOR_ID_PRIVY = resp["paymentConnectorId"]
    print(f"✅ Stripe Privy connector: {CONNECTOR_ID_PRIVY}")

    print("\nWaiting for Privy connector to reach READY...")
    wait_for_status(
        cp_client.get_payment_connector,
        expected_status="READY",
        poll_interval=5,
        timeout=120,
        paymentManagerId=MANAGER_ID,
        paymentConnectorId=CONNECTOR_ID_PRIVY,
    )
    print("✅ Privy connector is READY")

    write_env_updates({"CONNECTOR_ID_PRIVY": CONNECTOR_ID_PRIVY})
    print("💾 .env updated: CONNECTOR_ID_PRIVY")

### 4.5 Payment Instrument 4개 생성

현재 `EMBEDDED_CRYPTO_WALLET`만 사용 가능한 `paymentInstrumentType`입니다.
`linkedAccounts`는 필수이며 AgentCore Payments는 email을 사용하여 wallet을 소유하는
vendor 측 최종 사용자를 조회하거나
생성합니다.

Notebook은 connector별 두 개씩 총 **네 개**의 instrument를 생성합니다.

| Variable | Connector | `network` | 결제 Network |
|----------|-----------|-----------|--------|
| `PAYMENT_INSTRUMENT_ID_CDP_EVM`   | CoinbaseCDP | `ETHEREUM` | Base Sepolia |
| `PAYMENT_INSTRUMENT_ID_CDP_SOL`   | CoinbaseCDP | `SOLANA`   | Solana Devnet |
| `PAYMENT_INSTRUMENT_ID_PRIVY_EVM` | StripePrivy | `ETHEREUM` | Base Sepolia |
| `PAYMENT_INSTRUMENT_ID_PRIVY_SOL` | StripePrivy | `SOLANA`   | Solana Devnet |

네 instrument 모두 동일한 linked email 및 operator `USER_ID` header를
공유합니다. Service는 각각에 vendor 측 `userId`를 할당하며 §6에서는
`GetPaymentInstrument`를 통해 이를 읽고 이후 모든 data-plane 호출에
전달합니다.

Coinbase wallet의 response에는 **Coinbase Wallet Hub**의 `redirectUrl`이 포함됩니다.
이를 열어 linked email을 검증하고 USDC를 입금한 후 AgentCore Payments가 사용자를 대신해
sign할 수 있도록 signing delegation을
부여합니다.

Privy instrument는 `redirectUrl`을 반환하지 않습니다. §4.2b에서 등록한 Privy authorization
key가 wallet owner이지만 `ProcessPayment`가 작동하려면 최종 사용자가 wallet의 signer에
권한을 부여해야
합니다. Privy delegation flow는 별도로 다루며, flow가 연결될 때까지 §7에서
Privy row를 건너뜁니다.

작은 helper가 네 wallet 각각에 대해 동일한 CreatePaymentInstrument + wait +
GetPaymentInstrument cycle을 실행합니다.

#### 4.5a Coinbase CDP — ETHEREUM (Base Sepolia)


In [ ]:
def create_instrument(*, connector_id: str, network: str, env_key_id: str, env_key_wallet: str, label: str):
    """CreatePaymentInstrument 호출 후 ACTIVE 상태를 기다리고 walletAddress를 다시 조회합니다.

    (instrument_id, wallet_address, redirect_url)을 반환합니다.
    값을 확인하는 즉시 두 환경 변수 키를 저장합니다.
    """
    print(f"\n── Creating {label} instrument ({network}) ──")
    resp = dp_client_mgmt.create_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=connector_id,
        userId=USER_ID,
        paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
        paymentInstrumentDetails={
            "embeddedCryptoWallet": {
                "network": network,
                "linkedAccounts": [{"email": {"emailAddress": INSTRUMENT_EMAIL}}],
            }
        },
        clientToken=_client_token(),
    )
    instrument = resp["paymentInstrument"]
    instrument_id = instrument["paymentInstrumentId"]
    crypto = instrument["paymentInstrumentDetails"]["embeddedCryptoWallet"]
    wallet_address = crypto.get("walletAddress", "")
    redirect_url = crypto.get("redirectUrl")

    print(f"  paymentInstrumentId: {instrument_id}")
    print(f"  walletAddress:       {wallet_address or '(pending)'}")
    if redirect_url:
        print(f"  redirectUrl:         {redirect_url}")

    print(f"  Waiting for {label} instrument to become ACTIVE...")
    wait_for_status(
        dp_client_mgmt.get_payment_instrument,
        expected_status="ACTIVE",
        poll_interval=5,
        timeout=120,
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=instrument_id,
        userId=USER_ID,
    )
    refreshed = dp_client_mgmt.get_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=instrument_id,
        userId=USER_ID,
    )["paymentInstrument"]
    wallet_address = refreshed["paymentInstrumentDetails"]["embeddedCryptoWallet"].get("walletAddress", "")
    print(f"  ✅ {label} instrument ACTIVE")

    write_env_updates({env_key_id: instrument_id, env_key_wallet: wallet_address})
    print(f"💾 .env updated: {env_key_id}, {env_key_wallet}")
    return instrument_id, wallet_address, redirect_url


PAYMENT_INSTRUMENT_ID_CDP_EVM, WALLET_ADDRESS_CDP_EVM, REDIRECT_CDP_EVM = create_instrument(
    connector_id=CONNECTOR_ID_CDP,
    network="ETHEREUM",
    env_key_id="PAYMENT_INSTRUMENT_ID_CDP_EVM",
    env_key_wallet="WALLET_ADDRESS_CDP_EVM",
    label="CDP EVM",
)

print("\n" + "=" * 60)
print("  FUND THIS WALLET WITH BASE SEPOLIA USDC")
print("=" * 60)
print(f"  Address: {WALLET_ADDRESS_CDP_EVM or '(still provisioning)'}")
if REDIRECT_CDP_EVM:
    print(f"  Hub:     {REDIRECT_CDP_EVM}")
print("  Faucet:  https://faucet.circle.com/  (pick Base Sepolia)")
print("=" * 60)

##### Coinbase Wallet Hub에서 Signing Delegation 부여

Coinbase EMBEDDED_CRYPTO_WALLET을 생성할 때마다 이전 셀에 출력되는
Coinbase Wallet Hub의 `redirectUrl`이 반환됩니다. 이를 열고 다음 작업을 수행합니다.

1. `INSTRUMENT_EMAIL`로 설정한 email을 사용하여 **로그인**합니다. Hub에서 해당 address로
   one-time passcode(OTP)를 보냅니다.

<div style="text-align:left">
    <img src="images/cdp_hub_signin.png" alt="Coinbase Wallet Hub sign-in screen" width="75%"/>
</div>

2. Hub에 **OTP를 입력**합니다.

<div style="text-align:left">
    <img src="images/cdp_hub_otp.png" alt="Coinbase Wallet Hub OTP entry" width="75%"/>
</div>

3. Agent에 **signing delegation을 부여**합니다. 이 단계를 수행하지 않으면
   `ProcessPayment`가 *Delegated signing grant is not active*를 반환합니다.
4. Grant가 활성 상태로 유지되는 기간인 **delegation duration을 설정**합니다.
   활성 상태로 유지할 기간을 지정합니다.

<div style="text-align:left">
    <img src="images/cdp_hub_delegation.png" alt="Grant delegation with duration" width="75%"/>
</div>

5. 이전 셀에 출력된 **EVM wallet address를 복사**합니다.
6. 복사한 address를 사용하여 **Base Sepolia**의
   [Circle faucet](https://faucet.circle.com/)에서 **testnet USDC를 요청**합니다.
   복사한 address를 사용합니다.
7. Hub에서 **Solana로 전환**합니다.
8. Network dropdown에서 **Solana Devnet을 선택**합니다.
9. **Solana wallet address를 복사**합니다.
10. 복사한 Solana address를 사용하여 동일한 faucet의 **Solana Devnet**에서
    **testnet USDC를 요청**합니다.

두 wallet 모두에 balance가 표시되면 다음 섹션의 §4.5b를 실행하여 동일한 delegation 아래에
Solana instrument를 생성합니다.


#### 4.5b Coinbase CDP — SOLANA (Solana Devnet)


In [ ]:
PAYMENT_INSTRUMENT_ID_CDP_SOL, WALLET_ADDRESS_CDP_SOL, REDIRECT_CDP_SOL = create_instrument(
    connector_id=CONNECTOR_ID_CDP,
    network="SOLANA",
    env_key_id="PAYMENT_INSTRUMENT_ID_CDP_SOL",
    env_key_wallet="WALLET_ADDRESS_CDP_SOL",
    label="CDP SOLANA",
)

print("\n" + "=" * 60)
print("  FUND THIS WALLET WITH SOLANA DEVNET USDC")
print("=" * 60)
print(f"  Address: {WALLET_ADDRESS_CDP_SOL or '(still provisioning)'}")
if REDIRECT_CDP_SOL:
    print(f"  Hub:     {REDIRECT_CDP_SOL}")
print("  Faucet:  https://faucet.circle.com/  (pick Solana Devnet)")
print("=" * 60)

#### 4.5c Stripe via Privy — ETHEREUM (Base Sepolia)


In [ ]:
PAYMENT_INSTRUMENT_ID_PRIVY_EVM, WALLET_ADDRESS_PRIVY_EVM, REDIRECT_PRIVY_EVM = create_instrument(
    connector_id=CONNECTOR_ID_PRIVY,
    network="ETHEREUM",
    env_key_id="PAYMENT_INSTRUMENT_ID_PRIVY_EVM",
    env_key_wallet="WALLET_ADDRESS_PRIVY_EVM",
    label="Privy EVM",
)

print("\n" + "=" * 60)
print("  Privy EVM wallet created.")
print("=" * 60)
print(f"  Address: {WALLET_ADDRESS_PRIVY_EVM or '(still provisioning)'}")
print("  Note:    Grant signing delegation via the Privy Wallet Hub")
print("           frontend (see §4.5e below) before §7 can spend from")
print("           this wallet. Once delegation is granted, §7 spends here too.")
print("=" * 60)

#### 4.5d Stripe via Privy — SOLANA (Solana Devnet)


In [ ]:
PAYMENT_INSTRUMENT_ID_PRIVY_SOL, WALLET_ADDRESS_PRIVY_SOL, REDIRECT_PRIVY_SOL = create_instrument(
    connector_id=CONNECTOR_ID_PRIVY,
    network="SOLANA",
    env_key_id="PAYMENT_INSTRUMENT_ID_PRIVY_SOL",
    env_key_wallet="WALLET_ADDRESS_PRIVY_SOL",
    label="Privy SOLANA",
)

print("\n" + "=" * 60)
print("  Privy Solana wallet created.")
print("=" * 60)
print(f"  Address: {WALLET_ADDRESS_PRIVY_SOL or '(still provisioning)'}")
print("  Note:    Grant signing delegation via the Privy Wallet Hub frontend (§4.5e).")
print("=" * 60)

##### Privy Wallet Hub에서 Signing Delegation 부여

Privy embedded wallet에 대해 agent가 `ProcessPayment`를 호출하려면 먼저 별도의
**signing delegation**이 필요합니다. Delegation은 Privy와 AWS AgentCore Bedrock team이
공동으로 관리하는 작은 Next.js frontend인
다음 repository에서
부여합니다: [privy-io/aws-agentcore-sdk](https://github.com/privy-io/aws-agentcore-sdk).

아래 네 셀을 순서대로 실행합니다.

1. **Clone** — frontend를 `privy-delegation/`으로 가져옵니다.
2. **Generate env** — `.env`의 `PRIVY_APP_ID`, `PRIVY_APP_SECRET`,
   `PRIVY_AUTHORIZATION_ID`를 사용하여 `privy-delegation/.env.local`을
   작성하고 frontend를 **testnet**으로 고정하여 Base Sepolia + Solana Devnet
   balance를 읽도록 합니다.
3. **Install** — `npm install` 실행(idempotent)
4. **Start** — background에서 `npm run dev`를 시작하고 server가
   `http://localhost:3000`에서 실행될 때까지 대기(loopback 전용이므로
   local dev server에는 TLS 불필요)

[http://localhost:3000](http://localhost:3000)을 열고 다음 작업을 수행합니다.

1. `INSTRUMENT_EMAIL`로 설정한 email을 사용하여 **로그인**합니다. Privy에서 해당 address로
   one-time passcode(OTP)를 보냅니다.

<div style="text-align:left">
    <img src="images/privy_landing.png" alt="Privy Wallet Hub landing screen with email login" width="75%"/>
</div>
2. **Wallet을 확인합니다.** Hub는 Base Sepolia 및 Solana Devnet을 직접 query하고
   두 network의 USDC balance를 표시합니다.
3. **Agent에 access를 위임합니다.** Agent가 지출하도록 허용할 각 wallet에서
   **Delegate**를 선택합니다. 이 설정이 없으면 `ProcessPayment`가
   `Delegated signing grant is not active`를 반환합니다.

<div style="text-align:left">
    <img src="images/privy_give_access.png" alt="Granting agent signing access in the Privy Wallet Hub" width="75%"/>
</div>
4. Hub에서 **Receive**를 선택하여 wallet address를 복사합니다.
5. 복사한 address를 사용하여 **Base Sepolia** 또는 **Solana Devnet**의
   [Circle faucet](https://faucet.circle.com/)에서 **testnet USDC를 요청**합니다.
   Stripe를 통한 card funding은 mainnet 전용이며
   `NEXT_PUBLIC_NETWORK_MODE=testnet`일 때
   비활성화됩니다.

두 wallet 모두 balance가 표시되고 delegation이 부여되면 아래 stop 셀을 실행하여
frontend를 종료하고 §5를 계속할 수 있습니다.


In [ ]:
# 이 folder에 Privy Wallet Hub frontend가 없으면 clone
#
# Frontend는 upstream privy-io/aws-agentcore-sdk에 있으며 Privy와
# AWS AgentCore Bedrock team이 공동 관리함.
# 로컬에서 `privy-delegation/`으로 clone(upstream repo directory name은
# `aws-agentcore-sdk`이지만 이후 모든 셀이 동일한 path를 사용하도록
# 명시적 target 전달). Folder가 이미 있으면 이 셀의 재실행은
# 아무 작업도 하지 않음
import pathlib
import shutil
import subprocess

HUB_ROOT = pathlib.Path("privy-delegation").resolve()
PRIVY_REMOTE = "https://github.com/privy-io/aws-agentcore-sdk.git"

if HUB_ROOT.exists():
    print(f"↷ {HUB_ROOT.name}/ already present, skipping clone.")
else:
    if not shutil.which("git"):
        raise RuntimeError("git not found on PATH. Install git from https://git-scm.com/ and re-run this cell.")
    print(f"Cloning {PRIVY_REMOTE} into {HUB_ROOT.name}/ ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", PRIVY_REMOTE, str(HUB_ROOT)],
        check=True,
    )
    print("✅ Clone complete.")

In [ ]:
# Privy Wallet Hub frontend용 privy-delegation/.env.local 생성
#
# 상위 .env key를 frontend에서 요구하는 name에 매핑하고
# balance + transfer가 이 Notebook과 동일한 Base Sepolia 및 Solana Devnet을
# 대상으로 하도록 NETWORK_MODE=testnet으로 고정.
# 셀 재실행 시 .env.local을 idempotent하게 덮어씀
import pathlib

HUB_ROOT = pathlib.Path("privy-delegation").resolve()
HUB_ENV = HUB_ROOT / ".env.local"

missing_for_hub = [
    n
    for n, v in (
        ("PRIVY_APP_ID", PRIVY_APP_ID),
        ("PRIVY_APP_SECRET", PRIVY_APP_SECRET),
        ("PRIVY_AUTHORIZATION_ID", PRIVY_AUTHORIZATION_ID),
    )
    if not v
]
if missing_for_hub:
    raise RuntimeError(f"Cannot write {HUB_ENV} — missing in parent .env: {missing_for_hub}")

HUB_ENV.write_text(
    f"# Generated by pay-for-api.ipynb. Do not commit.\n"
    f"# Mapped from the parent .env so the Privy Wallet Hub frontend\n"
    f"# reads the same App ID / App Secret / Signer ID this notebook uses.\n"
    f"NEXT_PUBLIC_PRIVY_APP_ID={PRIVY_APP_ID}\n"
    f"PRIVY_APP_SECRET={PRIVY_APP_SECRET}\n"
    f"NEXT_PUBLIC_PRIVY_SIGNER_ID={PRIVY_AUTHORIZATION_ID}\n"
    f"NEXT_PUBLIC_NETWORK_MODE=testnet\n"
)
print(f"✅ Wrote {HUB_ENV}")
print(f"   App ID:    {PRIVY_APP_ID}")
print(f"   Signer ID: {PRIVY_AUTHORIZATION_ID}")
print(f"   Network:   testnet (Base Sepolia + Solana Devnet)")
print()
print("Next: run the install + start cells that follow to bring the hub up.")

In [ ]:
# Privy Wallet Hub frontend의 Node dependency 설치. npm은 Node.js와 함께
# 제공되므로 추가 package manager 설정 불필요.
# 이 셀은 idempotent하며 node_modules/가 이미 있으면 설치 생략
import pathlib
import shutil
import subprocess

HUB_ROOT = pathlib.Path("privy-delegation").resolve()

if not shutil.which("npm"):
    raise RuntimeError(
        "npm not found on PATH. Install Node.js (which includes npm) "
        "from https://nodejs.org/ or via your package manager."
    )

if (HUB_ROOT / "node_modules").exists():
    print(f"↷ {HUB_ROOT.name}/node_modules already present, skipping install.")
else:
    print(f"Running npm install in {HUB_ROOT}...")
    subprocess.run(["npm", "install"], cwd=HUB_ROOT, check=True)
    print("✅ npm install complete.")

In [ ]:
# Background에서 Privy Wallet Hub frontend를 시작하고 dev server가
# http://localhost:3000에서 응답할 때까지 대기. 아래 stop 셀에서 종료할 수 있도록
# PID를 HUB_DEV_PROCESS에 저장
import os
import pathlib
import socket
import subprocess
import time

HUB_ROOT = pathlib.Path("privy-delegation").resolve()
HUB_PORT = 3000
HUB_URL = f"http://localhost:{HUB_PORT}"


def _port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        try:
            s.connect((host, port))
            return True
        except (ConnectionRefusedError, socket.timeout, OSError):
            return False


# Port 3000에서 기존 dev server가 실행 중이면 재사용
already_running = _port_open("127.0.0.1", HUB_PORT)
if already_running:
    print(f"↷ Something is already listening on port {HUB_PORT}; reusing it.")
    HUB_DEV_PROCESS = None
else:
    log_path = HUB_ROOT / "npm-dev.log"
    log_handle = open(log_path, "w")
    HUB_DEV_PROCESS = subprocess.Popen(
        ["npm", "run", "dev"],
        cwd=HUB_ROOT,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        # 셀이 끝날 때 kernel이 dev server를 종료하지 않도록
        # 새 process group으로 detach
        start_new_session=True,
    )
    print(f"Started npm dev (PID {HUB_DEV_PROCESS.pid}); logs: {log_path}")
    print(f"Waiting for {HUB_URL} to come up...")
    deadline = time.time() + 90  # Next.js cold start에는 최대 1분이 걸릴 수 있음
    while time.time() < deadline:
        if _port_open("127.0.0.1", HUB_PORT):
            break
        if HUB_DEV_PROCESS.poll() is not None:
            raise RuntimeError(f"npm dev exited early (rc={HUB_DEV_PROCESS.returncode}); see {log_path} for details.")
        time.sleep(1)
    else:
        raise RuntimeError(f"npm dev did not respond on port {HUB_PORT} within 90s. See {log_path} for details.")
    print(f"✅ Hub is up at {HUB_URL}")

print()
print(f"Open {HUB_URL} in your browser, sign in with INSTRUMENT_EMAIL,")
print("delegate each wallet, fund via the Circle faucet, then run the")
print("STOP cell below to shut the dev server down.")

In [ ]:
# 위에서 시작한 Privy Wallet Hub frontend 종료. Start 셀에서 이미 실행 중인
# server를 재사용했다면 생략(HUB_DEV_PROCESS가 None)
import os
import signal

proc = globals().get("HUB_DEV_PROCESS")
if proc is None:
    print("↷ No background npm dev process to stop (server was reused).")
elif proc.poll() is not None:
    print(f"↷ npm dev already exited (rc={proc.returncode}).")
else:
    # Process group을 종료하여 npm wrapper와 Next.js dev server의
    # child worker까지 함께 종료
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        proc.wait(timeout=10)
    except Exception:
        os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    print(f"✅ Stopped npm dev (PID {proc.pid}).")

## 5. Payment Session 생성

**Payment session**은 agent 지출에 대해 budget과 시간이 제한된 authorization입니다.
Operator(`ManagementRole`을 사용하는 사용자)가 `maxSpendAmount`와
`expiryTimeInMinutes`를 설정합니다. 생성된 후 agent는 해당 budget 내에서만
지출할 수 있으며 limit을 늘리거나 session을
연장할 수 없습니다.

Session은 connector 또는 instrument가 아니라 Manager로 scope가 지정되므로
단일 session이 agent가 결제하는 모든 instrument를 포함합니다. Demo에서 CDP와 Privy의
spend ledger를 분리하도록 Notebook은 **provider별로 하나의
session**을 생성합니다. CDP session은 두 CDP instrument(EVM + SOL)를 포함하고
Privy session은 두 Privy instrument를 포함합니다.

참고 사항:

* `expiryTimeInMinutes`는 필수입니다(범위 15~480).
* `clientToken`은 33자 이상이어야 하며 UUID v4(36자)가 이 조건을 충족합니다.
* `userId`는 body field가 아니라 HTTP header입니다.

### 5.1 Session Client 구축


In [ ]:
from datetime import datetime

boto_session = Session(region_name=AWS_REGION)

# ── Management client(ManagementRole — session 생성) ──────────────────
mgmt_session = assume_role(
    boto_session,
    MANAGEMENT_ROLE_ARN,
    session_name="pay-for-api-mgmt-session",
)
dp_client = mgmt_session.client("bedrock-agentcore")

# ── Agent client(ProcessPaymentRole - 결제 서명) ─────────────────────
agent_session = assume_role(
    boto_session,
    PROCESS_PAYMENT_ROLE_ARN,
    session_name="pay-for-api-agent",
)
dp_agent_client = agent_session.client("bedrock-agentcore")

print("\n✅ Session clients ready")

### 5.2 Coinbase CDP session


In [ ]:
resp = dp_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    userId=USER_ID,
    expiryTimeInMinutes=SESSION_EXPIRY_MINUTES,
    limits={"maxSpendAmount": {"value": SESSION_MAX_SPEND, "currency": "USD"}},
    clientToken=_client_token(),
)
SESSION_ID_CDP = resp["paymentSession"]["paymentSessionId"]
print(f"✅ Coinbase CDP session created")
print(f"   Session ID: {SESSION_ID_CDP}")
print(f"   Budget:     ${SESSION_MAX_SPEND} USD (covers CDP EVM + SOL)")
print(f"   Expires in: {SESSION_EXPIRY_MINUTES} minutes")

write_env_updates({"SESSION_ID_CDP": SESSION_ID_CDP})
print("💾 .env updated: SESSION_ID_CDP")

### 5.3 Stripe via Privy session


In [ ]:
resp = dp_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    userId=USER_ID,
    expiryTimeInMinutes=SESSION_EXPIRY_MINUTES,
    limits={"maxSpendAmount": {"value": SESSION_MAX_SPEND, "currency": "USD"}},
    clientToken=_client_token(),
)
SESSION_ID_PRIVY = resp["paymentSession"]["paymentSessionId"]
print(f"✅ Stripe Privy session created")
print(f"   Session ID: {SESSION_ID_PRIVY}")
print(f"   Budget:     ${SESSION_MAX_SPEND} USD (covers Privy EVM + SOL)")
print(f"   Expires in: {SESSION_EXPIRY_MINUTES} minutes")

write_env_updates({"SESSION_ID_PRIVY": SESSION_ID_PRIVY})
print("💾 .env updated: SESSION_ID_PRIVY")

## 6. Fun Facts Agent 구축

이 사용 사례에서는 하나의 tool(`strands-agents-tools`의 `http_request`)과 특정
(instrument, session) pair로 구성된 `AgentCorePaymentsPlugin`을 사용하는
최소한의 Strands agent pattern **하나**를
구축합니다. Plugin의 `AgentCorePaymentsPluginConfig`는 단일
`payment_instrument_id` + `payment_session_id` + network preference를
고정하므로 서로 다른 network에서 결제할 때마다 동일한 factory를 사용하여
agent를 다시 구축합니다. §6에서는 동일한 seller에 대해 EVM에서 한 번,
Solana에서 한 번 실행합니다.

두 실행의 payment flow는 동일합니다.

1. Agent가 `http_request` `GET <seller>/facts?topic=<x>` 호출
2. Seller가 Base Sepolia와 Solana Devnet을 모두 포함하는 x402 `accepts` array와 함께
   **HTTP 402** 반환
3. `AgentCorePaymentsPlugin`이 intercept하고 해당 실행의 (manager, session, instrument, user)에 대해
   **`ProcessPayment`** 호출. Factory에 전달된 `network_preferences_config`가
   plugin이 `accepts[]`에서 먼저 선택할
   entry 결정
4. Seller가 x402 facilitator를 통해 검증 및 settlement를 수행하고 유료 fact와 함께
   **200 OK** 반환

Agent flow는 Coinbase CDP 또는 Stripe via Privy wallet provider에서도 동일하며
service가 적절한 signer를 선택합니다.

> Plugin은 runtime에 read-only management tool 세 개(`get_payment_instrument`,
> `list_payment_instruments`, `get_payment_session`)도 등록합니다.
> 아래 system prompt는 model에 `http_request`만 사용하도록 지시하며 plugin tool은
> operator 및 debug flow 전용입니다. 또한 plugin은 operator에게 실행 중 입력을 요청해야 하는
> agent를 위해 `PaymentInstrumentConfigurationRequired` /
> `PaymentSessionConfigurationRequired` interrupt contract를
> 노출합니다. `auto_payment=False`를 사용하는 human-in-the-loop flow를 포함한
> 전체 configuration surface는 [Strands SDK 참고 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-process-payment.html)를
> 참조하세요.
>


In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands_tools import http_request

from bedrock_agentcore.payments.integrations.config import (
    AgentCorePaymentsPluginConfig,
)
from bedrock_agentcore.payments.integrations.strands.plugin import (
    AgentCorePaymentsPlugin,
)

SYSTEM_PROMPT = (
    "You are a research agent backed by Amazon Bedrock AgentCore Payments. "
    "Your only tool is `http_request`. The AgentCorePaymentsPlugin watches "
    "every request and, when it sees an HTTP 402, silently calls "
    "ProcessPayment and retries with an `X-PAYMENT` header — you never "
    "handle private keys, assemble headers, or budget limits.\n"
    "\n"
    "The plugin also registers three read-only tools — "
    "`get_payment_instrument`, `list_payment_instruments`, "
    "`get_payment_session` — which the agent should not call. They exist for "
    "operator debug flows, not for you. Use only `http_request`.\n"
    "\n"
    "SELLER\n"
    "  Endpoint:  GET <seller>/facts?topic=<topic>\n"
    "  Topics:    space, oceans, ai, payments (anything else returns a "
    "random general fact)\n"
    "  Price:     $0.01 USDC per successful call\n"
    "  Response:  {'x402_content': {'data': '<JSON string>', ...}, "
    "'x402_meta': {...}}\n"
    "             `x402_content.data` is a JSON string; parse it to read "
    "`{'topic': ..., 'fact': ...}`.\n"
    "\n"
    "RULES\n"
    "  1. Make one `http_request` GET per topic the user asks about — if "
    "they ask for two, make two.\n"
    "  2. If the user names a topic outside the supported list, pick the "
    "closest supported one (e.g. volcanoes→space, whales→oceans) rather "
    "than letting the seller fall back silently.\n"
    "  3. Parse `x402_content.data` and quote the `fact` verbatim in your "
    "reply.\n"
    "  4. Always close with a one-line spend summary: "
    "`Total spent: $<0.01 × successful_calls>`.\n"
    "  5. If a call returns anything other than 200, explain the error in "
    "plain language and do not retry — the plugin already retried once."
)

# 공유 Bedrock model — cross-region US inference profile을 통한
# Claude Sonnet 4.5. Run별 state가 없으므로 실행 간에 동일한
# BedrockModel instance 재사용
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=AWS_REGION,
    temperature=0.7,
)


def resolve_payment_user_id(instrument_id: str) -> str:
    """지정한 결제 수단에서 paymentInstrument.userId를 가져옵니다.

    서비스는 CreatePaymentInstrument에서 이 값을 할당하며 이후 모든 작업
    (ProcessPayment, GetPaymentSession, GetPaymentInstrumentBalance)은 이 값을
    대상으로 해야 합니다. 요청 헤더의 운영자 USER_ID는 서비스가 기록하는 값과 다릅니다.
    """
    resp = dp_client.get_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=instrument_id,
        userId=USER_ID,
    )
    return resp["paymentInstrument"]["userId"]


def build_agent(*, instrument_id: str, session_id: str, network_preferences: list[str], label: str) -> Agent:
    """하나의 (instrument, session) 범위에 맞는 새 Strands 에이전트를 구성합니다.

    AgentCorePaymentsPluginConfig가 하나의 instrument와 session을 고정하므로
    여러 인스턴스를 유지하지 않고 실행마다 에이전트를 재구성합니다.
    `network_preferences`는 우선순위에 따른 CAIP-2 목록이며 seller의 402
    accepts[]에서 결제 수단 네트워크와 일치하는 항목을 선택하도록 알려 줍니다.
    """
    payment_user_id = resolve_payment_user_id(instrument_id)
    print(f"   {label} paymentUserId: {payment_user_id}")

    plugin = AgentCorePaymentsPlugin(
        config=AgentCorePaymentsPluginConfig(
            payment_manager_arn=MANAGER_ARN,
            user_id=payment_user_id,
            payment_instrument_id=instrument_id,
            payment_session_id=session_id,
            region=AWS_REGION,
            agent_name=f"pay-for-api-agent-{label.lower()}",
            network_preferences_config=network_preferences,
        )
    )
    return Agent(
        model=model,
        tools=[http_request],
        plugins=[plugin],
        system_prompt=SYSTEM_PROMPT,
    )


print("✅ Agent factory ready — §8 calls build_agent() once per network run")

## 7. Agent에 Fact 구매 요청(로컬 실행)

EVM agent를 먼저 실행한 다음 Solana agent를 실행합니다. Plugin은 agent가 수행하는
모든 HTTP request에 대해 해당 instrument가 대상으로 하는 network에서
402 → ProcessPayment → retry를 처리합니다. Log에서 각 agent가 구매하는 fact마다
`ProcessPayment` 호출 하나가 표시되어야 합니다.


In [ ]:
# 하나의 agent pattern을 (provider, network)별로 다시 구축. Plugin이
# (instrument, session, network preferences) triple을 고정하므로 각 실행은
# 해당 provider를 통해 해당 network에서 결제하는 역할만 수행하는
# 새 agent
#
# Privy wallet에는 Privy Wallet Hub frontend(§4.5e)를 통해 부여하는
# 별도 signing delegation이 필요. Delegation을 부여한 후 §7에서도
# Privy wallet에서 지출하도록 PRIVY_DELEGATION_WIRED_UP = True(기본값)로
# 설정. 개발 중 Privy row를 건너뛰려면 False로 변경
PRIVY_DELEGATION_WIRED_UP = True

runs = [
    {
        "label": "CDP EVM",
        "provider": "CoinbaseCDP",
        "network": "Base Sepolia",
        "instrument_id": PAYMENT_INSTRUMENT_ID_CDP_EVM,
        "session_id": SESSION_ID_CDP,
        "network_preferences": ["base-sepolia", "solana-devnet"],
        "topic": "space",
    },
    {
        "label": "CDP SOLANA",
        "provider": "CoinbaseCDP",
        "network": "Solana Devnet",
        "instrument_id": PAYMENT_INSTRUMENT_ID_CDP_SOL,
        "session_id": SESSION_ID_CDP,
        "network_preferences": ["solana-devnet", "base-sepolia"],
        "topic": "oceans",
    },
    {
        "label": "Privy EVM",
        "provider": "StripePrivy",
        "network": "Base Sepolia",
        "instrument_id": PAYMENT_INSTRUMENT_ID_PRIVY_EVM,
        "session_id": SESSION_ID_PRIVY,
        "network_preferences": ["base-sepolia", "solana-devnet"],
        "topic": "ai",
    },
    {
        "label": "Privy SOLANA",
        "provider": "StripePrivy",
        "network": "Solana Devnet",
        "instrument_id": PAYMENT_INSTRUMENT_ID_PRIVY_SOL,
        "session_id": SESSION_ID_PRIVY,
        "network_preferences": ["solana-devnet", "base-sepolia"],
        "topic": "payments",
    },
]

results = {}
for cfg in runs:
    label = cfg["label"]
    if not cfg["instrument_id"] or not cfg["session_id"]:
        print(f"\nℹ️  Skipping {label} — no instrument/session configured")
        continue
    if cfg["provider"] == "StripePrivy" and not PRIVY_DELEGATION_WIRED_UP:
        print(f"\nℹ️  Skipping {label} — Privy user delegation not yet wired up")
        continue
    print(f"\n── {label} run ({cfg['network']}) ──")
    agent = build_agent(
        instrument_id=cfg["instrument_id"],
        session_id=cfg["session_id"],
        network_preferences=cfg["network_preferences"],
        label=label.replace(" ", "-").lower(),
    )

    # 일시적인 Bedrock throttling 발생 시 retry. us-west-2의 Claude Sonnet 4.5는
    # back-to-back agent 실행이 겹치면 간헐적으로
    # ServiceUnavailableException("Too many connections") 반환
    # 짧은 exponential backoff로 Notebook을 실패시키지 않고
    # 처리
    import time
    from botocore.exceptions import ClientError

    attempt, max_attempts = 0, 4
    while True:
        try:
            results[label] = agent(
                f"Get me one interesting fact about {cfg['topic']} from the seller "
                f"at {SELLER_API_URL}. Tell me the total amount spent in USD at the end."
            )
            break
        except ClientError as exc:
            code = exc.response.get("Error", {}).get("Code", "")
            if code not in ("ServiceUnavailableException", "ThrottlingException"):
                raise
            attempt += 1
            if attempt >= max_attempts:
                raise
            wait = 2**attempt
            print(f"   ⚠️  Bedrock {code} — retry {attempt}/{max_attempts - 1} in {wait}s...")
            time.sleep(wait)
    print(f"\n── {label} response ──")
    print(results[label])

    # 이전 stream이 종료되는 동안 다음 agent가 동일한 Bedrock connection pool에
    # 겹치지 않도록 실행 간 잠시 대기
    time.sleep(2)

# 이후 셀(§8 Runtime invoke)이 계속 작동하도록 `result`에 기존 single-run
# variable name 유지
result = results.get("CDP EVM") or next(iter(results.values()), None)

## 8. AgentCore Runtime에 Agent 배포

§7에서 실행한 agent는 laptop에서 작동합니다. Managed service로 실행하려면 동일한
`Agent()` construction을 container 내부에 포함하여 **Amazon Bedrock AgentCore Runtime**에
배포합니다. 이 사용 사례의 `agent/` folder에는 다음 항목이
있습니다.

- `agent/container/` — FastAPI wrapper 및 동일한 agent 코드
- `agent/cdk/` — Docker를 통해 image를 로컬에서 build하고 CDK bootstrap ECR에 push한 후
  해당 image를 가리키는 AgentCore Runtime을 provision하는
  CDK stack

Stack은 Claude Sonnet 4.5 inference profile의 `bedrock:InvokeModel*` 및 plugin에 필요한
일부 AgentCore Payments data-plane operation(`ProcessPayment`, `GetPaymentSession`,
`GetPaymentInstrument`, `GetPaymentInstrumentBalance`, `GetResourcePaymentToken`)을
허용하는 최소 IAM role도
연결합니다. Control-plane operation(`CreatePaymentManager` 등)은 Runtime에
허용되지 않으며 operator가 보유합니다.

이 섹션을 실행하려면 다음 항목이 필요합니다.

- 설치된 **AWS CDK v2**(`npm install -g aws-cdk`) — 필요한 유일한 local tool이며
  container image는 **AWS CodeBuild**에서 build되므로 laptop에
  Docker가 필요하지 않습니다.

> ⚠️ **비용 안내:** 이 단계에서는 Amazon ECR repository, AWS CodeBuild build,
> AgentCore Runtime, AgentCore Memory resource, 이를 지원하는 CloudWatch log group과
> X-Ray trace를 provision합니다. 분당 과금되는 CodeBuild와 invocation별 과금되는
> Runtime은 stack을 계속 실행하면 비용이 누적될 수 있습니다.
> 작업을 마치면 §10을 실행하여
> 모든 항목을 제거하세요.

In [ ]:
# test/integration/deploy-agent.sh를 통해 agent runtime 배포
# Script가 CDK venv를 생성하고 CDK dependency를 설치하며 필요하면
# CDK를 bootstrap하고 Docker image를 build한 후 `cdk deploy` 실행
# 보통 몇 분 걸리는 build가 아무 출력 없이 진행되지 않도록 stdout streaming
import json
import subprocess
from pathlib import Path

HERE = Path(".").resolve()
AGENT_CDK_DIR = HERE / "agent" / "cdk"
DEPLOY = HERE / "test" / "integration" / "deploy-agent.sh"

if not DEPLOY.exists():
    raise FileNotFoundError(f"Expected {DEPLOY} — run this notebook from the 02-use-cases/01-pay-for-api directory.")

proc = subprocess.Popen(
    ["bash", str(DEPLOY)],
    cwd=str(HERE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"deploy-agent.sh failed with exit code {rc}")

cdk_outputs = json.loads((AGENT_CDK_DIR / "outputs.json").read_text())
outputs = cdk_outputs["AgentCorePaymentsBuyerAgentStack"]
AGENT_RUNTIME_ARN = outputs["AgentRuntimeArn"]
AGENT_RUNTIME_ID = outputs["AgentRuntimeId"]
print("\n✅ Runtime deployed")
print(f"   Runtime ARN:  {AGENT_RUNTIME_ARN}")
print(f"   Runtime ID:   {AGENT_RUNTIME_ID}")

### Transaction Search 활성화(일회성 Console 단계)

Runtime을 호출하기 전에 runtime page의 **Observability** tab에 trace 및 span detail이
표시되도록 Transaction Search를
활성화합니다. Runtime별 일회성 설정입니다.

1. **Amazon Bedrock AgentCore** console → **Runtime**을 엽니다.
2. 목록에서 **`pay_for_api_agent_runtime`**을 선택합니다.

<div style="text-align:left">
    <img src="images/agentcore_runtime_selected.png" alt="Selecting the agent runtime in the list" width="75%"/>
</div>

3. **Log deliveries and tracing** tab을 엽니다.
4. **Transaction Search**를 활성화합니다.
5. **Save**를 선택합니다.

<div style="text-align:left">
    <img src="images/agentcore_tracing_enable_section.png" alt="Enable Transaction Search and tracing" width="75%"/>
</div>


저장 후 panel에서 Transaction Search가 활성화되었는지 확인합니다.

<div style="text-align:left">
    <img src="images/agentcore_transaction_search_enabled.png" alt="Transaction Search enabled confirmation" width="75%"/>
</div>


> Transaction Search는 저장 후 indexing을 시작하는 데 몇 분이 걸립니다.
> Observability tab에 data가 바로 표시되지 않으면 잠시 기다렸다가
> 새로 고침하세요.


In [ ]:
# 배포된 runtime을 (provider, network)별로 한 번씩 호출. §7의 local pattern과
# 동일한 네 번의 실행 matrix. Container는 전달된 payload를 기반으로 invocation별
# Strands Agent + plugin을 생성하므로 서로 다른 ID를 선택하여
# 모든 wallet을 target으로 지정 가능
#
# Privy wallet에는 Privy에서 부여한 signing delegation이 필요함
# Wallet Hub frontend(4.5e절)에서 delegation을 설정하면 배포된
# runtime도 Privy instrument에 대해 ProcessPayment 호출 가능
import json as _json
import time as _time

PRIVY_DELEGATION_WIRED_UP = True

runtime_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

runtime_runs = [
    {
        "label": "CDP EVM",
        "provider": "CoinbaseCDP",
        "instrument_id": PAYMENT_INSTRUMENT_ID_CDP_EVM,
        "session_id": SESSION_ID_CDP,
        "network_preferences": ["base-sepolia", "solana-devnet"],
        "topic": "space",
    },
    {
        "label": "CDP SOLANA",
        "provider": "CoinbaseCDP",
        "instrument_id": PAYMENT_INSTRUMENT_ID_CDP_SOL,
        "session_id": SESSION_ID_CDP,
        "network_preferences": ["solana-devnet", "base-sepolia"],
        "topic": "oceans",
    },
    {
        "label": "Privy EVM",
        "provider": "StripePrivy",
        "instrument_id": PAYMENT_INSTRUMENT_ID_PRIVY_EVM,
        "session_id": SESSION_ID_PRIVY,
        "network_preferences": ["base-sepolia", "solana-devnet"],
        "topic": "ai",
    },
    {
        "label": "Privy SOLANA",
        "provider": "StripePrivy",
        "instrument_id": PAYMENT_INSTRUMENT_ID_PRIVY_SOL,
        "session_id": SESSION_ID_PRIVY,
        "network_preferences": ["solana-devnet", "base-sepolia"],
        "topic": "payments",
    },
]

remote_results = {}
for cfg in runtime_runs:
    label = cfg["label"]
    if not cfg["instrument_id"] or not cfg["session_id"]:
        print(f"\n\u21b7 {label}: skipped (no instrument/session)")
        continue
    if cfg["provider"] == "StripePrivy" and not PRIVY_DELEGATION_WIRED_UP:
        print(f"\n\u21b7 {label}: skipped (Privy user delegation not wired up)")
        continue

    print(f"\n\u2500\u2500 {label} runtime invoke \u2500\u2500")
    payment_user_id = resolve_payment_user_id(cfg["instrument_id"])

    resp = runtime_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        qualifier="DEFAULT",
        payload=_json.dumps(
            {
                "prompt": (
                    f"Get me one interesting fact about {cfg['topic']}. "
                    "Tell me the total amount spent in USD at the end."
                ),
                "sellerUrl": SELLER_API_URL,
                "managerArn": MANAGER_ARN,
                "instrumentId": cfg["instrument_id"],
                "sessionId": cfg["session_id"],
                "paymentUserId": payment_user_id,
                "networkPreferences": cfg["network_preferences"],
                "region": AWS_REGION,
            }
        ).encode(),
    )

    body = resp["response"]
    payload = b"".join(body.iter_chunks()) if hasattr(body, "iter_chunks") else body.read()
    remote_results[label] = _json.loads(payload)
    response_text = remote_results[label].get("response", remote_results[label])
    print(response_text)

    # 이전 stream이 종료되는 동안 runtime이 다음 호출을 queue하지 않도록
    # invocation 간 잠시 대기
    _time.sleep(2)

### Console에서 Runtime 검사

Runtime이 응답한 후 AgentCore console을 열어 내부 span과 log를
확인합니다.

1. Runtime detail page의 **Observability** 섹션에서 **View dashboard**를
   선택합니다.

<div style="text-align:left">
    <img src="images/agentcore_runtime_observability_section.png" alt="Observability section with View dashboard button" width="75%"/>
</div>

2. CloudWatch GenAI Observability dashboard가 열립니다. **Sessions** tab을 열고
   가장 최근 **Session ID**를 선택합니다.

<div style="text-align:left">
    <img src="images/agentcore_cloudwatch_genai_observability_dashboard.png" alt="CloudWatch GenAI Observability dashboard, Sessions tab" width="75%"/>
</div>

3. 가장 최근 **Trace ID**를 선택하여 `POST /invocations` event를 살펴봅니다.
   Model invocation, tool 호출, payment requirement(`402`), `ProcessPayment` span,
   **200 OK**를 반환하는 최종 retry를
   확인할 수 있습니다.

<div style="text-align:left">
    <img src="images/agentcore_observability_results.png" alt="Trace exploration showing POST /invocations events" width="75%"/>
</div>



## 9. Data Plane 검사

Agent가 지출한 후 read-only data-plane API를 살펴보며 service에 기록된 내용을
정확히 확인합니다. 여기의 모든 작업은 **management** client로 실행됩니다.
Agent의 `ProcessPaymentRole`에는 해당 API에 명시적 `Deny`가 적용되며 이를 통해
audit boundary를 적용합니다.

### 9.1 `GetPaymentSession` — Session State 및 남은 Budget


In [ ]:
# 전체 inspect 호출을 try/except로 감싸 STS session의 1시간 lifetime이 지나
# ExpiredTokenException이 발생할 때 raw boto3 traceback 대신 명확한
# 재실행 안내 표시
from botocore.exceptions import ClientError

try:
    # GetPaymentSession은 session별 실행 중인 budget state를 제공
    # Provider별 session 하나를 생성했으므로(§5) 둘 다 검사
    sessions_to_inspect = []
    if SESSION_ID_CDP:
        sessions_to_inspect.append(("CDP", SESSION_ID_CDP))
    if SESSION_ID_PRIVY:
        sessions_to_inspect.append(("Privy", SESSION_ID_PRIVY))

    for label, session_id in sessions_to_inspect:
        resp = dp_client.get_payment_session(
            paymentManagerArn=MANAGER_ARN,
            paymentSessionId=session_id,
            userId=USER_ID,
        )
        s = resp.get("paymentSession", {})
        budget = s.get("limits", {}).get("maxSpendAmount", {})
        avail = s.get("availableLimits", {}).get("availableSpendAmount", {})
        budget_val = budget.get("value", "?")
        budget_cur = budget.get("currency", "")
        avail_val = avail.get("value", "?")
        avail_cur = avail.get("currency", "")
        print(f"\n── {label} session ──")
        print(f"  Session ID:   {s.get('paymentSessionId', session_id)}")
        print(f"  Budget:       {budget_val} {budget_cur}")
        print(f"  Remaining:    {avail_val} {avail_cur}")
        print(f"  Expires in:   {s.get('expiryTimeInMinutes', '?')} minutes")
        print(f"  Created at:   {s.get('createdAt', '?')}")
except ClientError as _exc:
    if _exc.response.get("Error", {}).get("Code") == "ExpiredTokenException":
        print(
            "⏳  STS session credentials expired.\n"
            "   Re-run the §5.1 cell (Build session clients) to refresh "
            "`dp_client` / `dp_agent_client`,\n"
            "   then re-run THIS cell. (The fresh clients come up with "
            "auto-refreshing creds\n"
            "   via utils.assume_role, so this won't happen again in "
            "this kernel.)"
        )
        raise SystemExit("Re-run §5.1 (Build session clients), then re-run this cell.") from _exc
    raise

### 9.2 `GetPaymentInstrumentBalance` — On-chain USDC Balance

`GetPaymentInstrumentBalance`는 AgentCore Payments에 instrument wallet의 on-chain
USDC balance를 요청합니다. AgentCore Payments는 instrument network를
`chain`에 매핑합니다.

| `CryptoWalletNetwork` | `chain` (`BlockchainChainId`) |
|-----------------------|-------------------------------|
| `ETHEREUM`            | `BASE_SEPOLIA` (test) / `BASE` |
| `SOLANA`              | `SOLANA_DEVNET` / `SOLANA`     |

현재 token은 항상 `USDC`이며 `InstrumentBalanceToken` enum에서 허용하는 유일한
값입니다. Response의 `tokenBalance.amount`는 atomic amount입니다.
USDC는 소수점 6자리를 사용합니다.

> Higher-level SDK helper는 더 짧은 `{amount, currency}` shape를 반환할 수 있습니다.
> 이 Notebook에서 사용하는 direct boto3 호출은 더 완전한 `tokenBalance` structure를
> 반환합니다. 둘 다 올바르며 사용 가능한 경우 SDK helper를
> 사용하는 것이 좋습니다.


In [ ]:
# 전체 inspect 호출을 try/except로 감싸 STS session의 1시간 lifetime이 지나
# ExpiredTokenException이 발생할 때 raw boto3 traceback 대신 명확한
# 재실행 안내 표시
from botocore.exceptions import ClientError

try:
    # GetPaymentInstrumentBalance에는 (paymentManagerArn, paymentConnectorId,
    # paymentInstrumentId, chain, token)이 필요. Chain은 enum이며 여기서는
    # BASE_SEPOLIA 또는 SOLANA_DEVNET이고 token은 모두 USDC.
    # 실제 생성된 모든 (provider, network) instrument 순회
    wallets = [
        ("CDP EVM", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_EVM, WALLET_ADDRESS_CDP_EVM, "BASE_SEPOLIA"),
        ("CDP SOLANA", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_SOL, WALLET_ADDRESS_CDP_SOL, "SOLANA_DEVNET"),
        ("Privy EVM", CONNECTOR_ID_PRIVY, PAYMENT_INSTRUMENT_ID_PRIVY_EVM, WALLET_ADDRESS_PRIVY_EVM, "BASE_SEPOLIA"),
        (
            "Privy SOLANA",
            CONNECTOR_ID_PRIVY,
            PAYMENT_INSTRUMENT_ID_PRIVY_SOL,
            WALLET_ADDRESS_PRIVY_SOL,
            "SOLANA_DEVNET",
        ),
    ]

    for label, connector_id, instrument_id, wallet_address, chain in wallets:
        if not instrument_id or not connector_id:
            print(f"\n↷ {label}: skipped (no instrument)")
            continue
        payment_user_id = resolve_payment_user_id(instrument_id)
        resp = dp_client.get_payment_instrument_balance(
            paymentManagerArn=MANAGER_ARN,
            paymentConnectorId=connector_id,
            paymentInstrumentId=instrument_id,
            userId=payment_user_id,
            chain=chain,
            token="USDC",
        )
        # Atomic amount(token이 on-chain에서 사용하는 나눌 수 없는 최소 단위, USDC의 atomic unit 하나는 0.000001 USDC)을 사람이 읽을 수 있는 token amount로 변환
        # `amount`는 base unit 문자열(예: "19990000")이며 10**decimals로 나누어
        # USDC amount(예: 19.99) 계산
        bal = resp.get("tokenBalance", {})
        amount = int(bal.get("amount", "0")) / (10 ** int(bal.get("decimals", 6)))
        print(f"\n── {label} ({chain}) ──")
        print(f"  Wallet:  {wallet_address}")
        print(f"  Balance: {amount:.6f} {bal.get('token', 'USDC')}")
except ClientError as _exc:
    if _exc.response.get("Error", {}).get("Code") == "ExpiredTokenException":
        print(
            "⏳  STS session credentials expired.\n"
            "   Re-run the §5.1 cell (Build session clients) to refresh "
            "`dp_client` / `dp_agent_client`,\n"
            "   then re-run THIS cell. (The fresh clients come up with "
            "auto-refreshing creds\n"
            "   via utils.assume_role, so this won't happen again in "
            "this kernel.)"
        )
        raise SystemExit("Re-run §5.1 (Build session clients), then re-run this cell.") from _exc
    raise

### 9.3 `ListPaymentInstruments` — Manager 아래의 Instrument

`ListPaymentInstruments`는 지정한 `paymentManagerArn` 아래에서 caller가 볼 권한이 있는
모든 instrument를 나열합니다. Filter는 다음과 같습니다.

- `paymentConnectorId`(선택 사항) — 단일 connector로 범위 제한
- `userId`(header, 실제로는 필수 — USER_ID 대신 §4.5의 instrument summary에서
  vendor가 할당한 userId를 전달) — 단일 vendor-level user로
  범위 제한

Response는 `PaymentInstrumentSummary` object 목록입니다. 각 summary의 `userId`를
확인하세요. Coinbase wallet의 경우 service가 할당한 CDP 최종 사용자 UUID이며
이 값을 이후 모든 operation에 전달합니다.


In [ ]:
# 전체 inspect 호출을 try/except로 감싸 STS session의 1시간 lifetime이 지나
# ExpiredTokenException이 발생할 때 raw boto3 traceback 대신 명확한
# 재실행 안내 표시
from botocore.exceptions import ClientError

try:
    # ListPaymentInstruments는 Manager 아래의 모든 instrument를 나열
    # Summary에는 walletAddress/network가 없으므로 각 entry를
    # GetPaymentInstrument로 hydrate. Provider별 목록 범위를 자체 instrument로
    # 제한하도록 paymentConnectorId도 전달
    for label, connector_id, instrument_id in (
        ("CDP", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_EVM),
        ("Privy", CONNECTOR_ID_PRIVY, PAYMENT_INSTRUMENT_ID_PRIVY_EVM),
    ):
        if not instrument_id or not connector_id:
            print(f"\n\u21b7 {label}: skipped (no instrument)")
            continue
        payment_user_id = resolve_payment_user_id(instrument_id)
        resp = dp_client.list_payment_instruments(
            paymentManagerArn=MANAGER_ARN,
            paymentConnectorId=connector_id,
            userId=payment_user_id,
            maxResults=20,
        )
        summaries = resp.get("paymentInstruments", [])
        print(f"\n\u2500\u2500 {label} instruments ({len(summaries)}) \u2500\u2500")
        for i, summary in enumerate(summaries, start=1):
            # 각 entry를 hydrate하여 walletAddress + network 가져오기
            full = dp_client.get_payment_instrument(
                paymentManagerArn=MANAGER_ARN,
                paymentInstrumentId=summary["paymentInstrumentId"],
                userId=payment_user_id,
            )["paymentInstrument"]
            crypto = full.get("paymentInstrumentDetails", {}).get("embeddedCryptoWallet", {})
            address = crypto.get("walletAddress", "(no address)")
            network = crypto.get("network", "?")
            status = full.get("status", "?")
            print(f"  [{i}] {full['paymentInstrumentId']}  ({network}, {status})\n      Wallet: {address}")
except ClientError as _exc:
    if _exc.response.get("Error", {}).get("Code") == "ExpiredTokenException":
        print(
            "⏳  STS session credentials expired.\n"
            "   Re-run the §5.1 cell (Build session clients) to refresh "
            "`dp_client` / `dp_agent_client`,\n"
            "   then re-run THIS cell. (The fresh clients come up with "
            "auto-refreshing creds\n"
            "   via utils.assume_role, so this won't happen again in "
            "this kernel.)"
        )
        raise SystemExit("Re-run §5.1 (Build session clients), then re-run this cell.") from _exc
    raise

### 9.4 `ListPaymentSessions` — Manager 아래의 Session

`ListPaymentSessions`는 session에 대해 동일하게 작동합니다.
`paymentManagerArn` + 선택적 `userId` header로 scope가 지정됩니다. Summary에는
`expiryTimeInMinutes` 및 `createdAt` / `updatedAt` timestamp가 포함되어
각 session을 개별적으로 가져오지 않고도 audit view를 구축할 수 있습니다.


In [ ]:
# 전체 inspect 호출을 try/except로 감싸 STS session의 1시간 lifetime이 지나
# ExpiredTokenException이 발생할 때 raw boto3 traceback 대신 명확한
# 재실행 안내 표시
from botocore.exceptions import ClientError

try:
    # ListPaymentSessions는 session에 대해 동일하게 작동. Summary에는
    # `limits`와 `availableLimits`가 없으므로 한눈에 보는 audit view를 위해
    # GetPaymentSession을 통해 hydrate
    for label, instrument_id in (
        ("CDP", PAYMENT_INSTRUMENT_ID_CDP_EVM),
        ("Privy", PAYMENT_INSTRUMENT_ID_PRIVY_EVM),
    ):
        if not instrument_id:
            print(f"\n\u21b7 {label}: skipped (no instrument)")
            continue
        payment_user_id = resolve_payment_user_id(instrument_id)
        resp = dp_client.list_payment_sessions(
            paymentManagerArn=MANAGER_ARN,
            userId=payment_user_id,
            maxResults=20,
        )
        summaries = resp.get("paymentSessions", [])
        print(f"\n\u2500\u2500 {label} sessions ({len(summaries)}) \u2500\u2500")
        for i, summary in enumerate(summaries, start=1):
            full = dp_client.get_payment_session(
                paymentManagerArn=MANAGER_ARN,
                paymentSessionId=summary["paymentSessionId"],
                userId=payment_user_id,
            )["paymentSession"]
            budget = full.get("limits", {}).get("maxSpendAmount", {})
            avail = full.get("availableLimits", {}).get("availableSpendAmount", {})
            print(
                f"  [{i}] {full['paymentSessionId']}\n"
                f"      Budget:    {budget.get('value', '?')} {budget.get('currency', '')}\n"
                f"      Remaining: {avail.get('value', '?')} {avail.get('currency', '')}\n"
                f"      Expires:   {full.get('expiryTimeInMinutes', '?')} min\n"
                f"      Created:   {full.get('createdAt', '?')}"
            )
except ClientError as _exc:
    if _exc.response.get("Error", {}).get("Code") == "ExpiredTokenException":
        print(
            "⏳  STS session credentials expired.\n"
            "   Re-run the §5.1 cell (Build session clients) to refresh "
            "`dp_client` / `dp_agent_client`,\n"
            "   then re-run THIS cell. (The fresh clients come up with "
            "auto-refreshing creds\n"
            "   via utils.assume_role, so this won't happen again in "
            "this kernel.)"
        )
        raise SystemExit("Re-run §5.1 (Build session clients), then re-run this cell.") from _exc
    raise

## 10. 리소스 정리

이 사용 사례에서는 다음 유료 AWS resource를 provision했습니다. 이 섹션의 셀을
순서대로 실행하여 제거하고 추가 요금 발생을
중단합니다.

| Resource | 생성 위치 | 정리 방법 |
|----------|------------|---------------|
| Fun Facts seller(Amazon API Gateway HTTP API + AWS Lambda 함수) | §3 | §10 *Seller stack 제거* |
| AgentCore Runtime + Amazon ECR repository + AWS CodeBuild project | §8 | §10 *Agent runtime 제거* |
| AgentCore Memory resource | §8 | §10 *Agent runtime 제거*(agent stack과 함께 삭제) |
| Payment Manager, Connector, Instrument, Session, Credential Provider | §4 + §5 | §10 *AgentCore Payments resource 제거* |
| CloudWatch log group + X-Ray trace | §7 + §8에서 생성 | 유지됨 — 과거 log를 지워야 하면 console에서 수동 삭제 |
| IAM role 4개(`AgentCorePayments*Role`) | §2의 `setup-roles.sh` | 유지됨 — 상시 비용 없음. 완전히 정리하려면 각각 `aws iam delete-role` 실행 |

### Payment Session 취소

`DeletePaymentSession`은 server 측에서 session을 hard-delete합니다. Record는 영구적으로
제거되며 복구할 수 없습니다. Agent가 더 이상 지출하지 못하게 할 session의
revoke path입니다.


In [ ]:
import botocore.exceptions


def _safe_delete(fn, label: str, **kwargs) -> None:
    try:
        fn(**kwargs)
        print(f"  ✅ Deleted: {label}")
    except botocore.exceptions.ClientError as exc:
        code = exc.response["Error"]["Code"]
        msg = exc.response["Error"].get("Message", "")
        # 일부 cleanup path는 parent resource가 이미 제거된 경우 message에 "not found"가
        # 포함된 AccessDenied를 반환. ResourceNotFoundException과 동일한
        # 무해한 no-op으로 처리
        if code == "ResourceNotFoundException" or (code == "AccessDeniedException" and "not found" in msg.lower()):
            print(f"  ⚠️  Not found: {label}")
        elif code == "ExpiredTokenException":
            print(
                "⏳  STS session credentials expired.\n"
                "   Re-run the §4.1 cell (Assume roles + build clients) to "
                "refresh `dp_client_mgmt`,\n"
                "   then re-run THIS cell. (Fresh clients come up with "
                "auto-refreshing creds via\n"
                "   utils.assume_role, so this won't happen again in this "
                "kernel.)"
            )
            raise SystemExit("Re-run §4.1 (Assume roles + build clients), then re-run this cell.") from exc
        else:
            raise


if not MANAGER_ARN:
    print("ℹ️  Nothing to tear down — MANAGER_ARN is unset.")
else:
    # 1. 먼저 모든 instrument soft-delete. Manager / Connector cleanup에는
    #    ACTIVE instrument가 없어야 함
    for label, connector_id, instrument_id in (
        ("CDP EVM", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_EVM),
        ("CDP SOLANA", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_SOL),
        ("Privy EVM", CONNECTOR_ID_PRIVY, PAYMENT_INSTRUMENT_ID_PRIVY_EVM),
        ("Privy SOLANA", CONNECTOR_ID_PRIVY, PAYMENT_INSTRUMENT_ID_PRIVY_SOL),
    ):
        if not instrument_id or not connector_id:
            continue
        payment_user_id = resolve_payment_user_id(instrument_id)
        _safe_delete(
            dp_client_mgmt.delete_payment_instrument,
            f"Instrument {label} ({instrument_id})",
            paymentManagerArn=MANAGER_ARN,
            paymentConnectorId=connector_id,
            paymentInstrumentId=instrument_id,
            userId=payment_user_id,
        )

    # 2. 커넥터
    for label, connector_id in (
        ("CDP", CONNECTOR_ID_CDP),
        ("Privy", CONNECTOR_ID_PRIVY),
    ):
        if not connector_id:
            continue
        _safe_delete(
            cp_client.delete_payment_connector,
            f"{label} connector {connector_id}",
            paymentManagerId=MANAGER_ID,
            paymentConnectorId=connector_id,
            clientToken=_client_token(),
        )

    # 3. 관리자
    _safe_delete(
        cp_client.delete_payment_manager,
        f"Manager {MANAGER_ID}",
        paymentManagerId=MANAGER_ID,
        clientToken=_client_token(),
    )

    # 4. 자격 증명 공급자
    for label, name in (
        ("CDP", os.environ.get("CRED_PROVIDER_NAME_CDP", "")),
        ("Privy", os.environ.get("CRED_PROVIDER_NAME_PRIVY", "")),
    ):
        if not name:
            continue
        _safe_delete(
            cred_client.delete_payment_credential_provider,
            f"{label} Credential Provider {name}",
            name=name,
        )
    # 이후 Notebook 실행에서 방금 정리한 ID를 재사용하지 않도록
    # .env에서 삭제
    from utils import write_env_updates

    write_env_updates(
        {
            "MANAGER_ID": "",
            "MANAGER_ARN": "",
            "CRED_PROVIDER_NAME_CDP": "",
            "CREDENTIAL_PROVIDER_ARN_CDP": "",
            "CRED_PROVIDER_NAME_PRIVY": "",
            "CREDENTIAL_PROVIDER_ARN_PRIVY": "",
            "CONNECTOR_ID_CDP": "",
            "CONNECTOR_ID_PRIVY": "",
            "PAYMENT_INSTRUMENT_ID_CDP_EVM": "",
            "WALLET_ADDRESS_CDP_EVM": "",
            "PAYMENT_INSTRUMENT_ID_CDP_SOL": "",
            "WALLET_ADDRESS_CDP_SOL": "",
            "PAYMENT_INSTRUMENT_ID_PRIVY_EVM": "",
            "WALLET_ADDRESS_PRIVY_EVM": "",
            "PAYMENT_INSTRUMENT_ID_PRIVY_SOL": "",
            "WALLET_ADDRESS_PRIVY_SOL": "",
            "SESSION_ID_CDP": "",
            "SESSION_ID_PRIVY": "",
        }
    )
    print("\n✅ AgentCore Payments resources cleaned up.")
    print("💾 .env cleared of Manager/Connector/Instrument/Session IDs.")

### Seller Stack 제거

> ⚠️ **경고:** 사용 사례를 마친 후에만 실행하세요. Seller를 제거하면 `.env`의
> `SELLER_API_URL`이 무효화됩니다.


In [ ]:
!bash test/integration/destroy-seller.sh

### Agent Runtime 제거

§8에서 agent를 AgentCore Runtime에 배포한 경우에만 실행합니다. Agent를 로컬에서만
실행했다면 건너뜁니다.


In [ ]:
!bash test/integration/destroy-agent.sh

### AgentCore Payments Resource 제거

§4의 설정을 실행했고 생성한 모든 항목을 삭제하려면 아래 셀을 실행합니다.
순서가 중요합니다. 두 Instrument soft-delete → Connector → Manager →
Credential Provider 순서로 진행합니다.


In [ ]:
import botocore.exceptions


def _safe_delete(fn, label: str, **kwargs) -> None:
    try:
        fn(**kwargs)
        print(f"  ✅ Deleted: {label}")
    except botocore.exceptions.ClientError as exc:
        code = exc.response["Error"]["Code"]
        msg = exc.response["Error"].get("Message", "")
        # 일부 cleanup path는 parent resource가 이미 제거된 경우 message에 "not found"가
        # 포함된 AccessDenied를 반환. ResourceNotFoundException과 동일한
        # 무해한 no-op으로 처리
        if code == "ResourceNotFoundException" or (code == "AccessDeniedException" and "not found" in msg.lower()):
            print(f"  ⚠️  Not found: {label}")
        elif code == "ExpiredTokenException":
            print(
                "⏳  STS session credentials expired.\n"
                "   Re-run the §4.1 cell (Assume roles + build clients) to "
                "refresh `dp_client_mgmt`,\n"
                "   then re-run THIS cell. (Fresh clients come up with "
                "auto-refreshing creds via\n"
                "   utils.assume_role, so this won't happen again in this "
                "kernel.)"
            )
            raise SystemExit("Re-run §4.1 (Assume roles + build clients), then re-run this cell.") from exc
        else:
            raise


if not MANAGER_ARN:
    print("ℹ️  Nothing to tear down — MANAGER_ARN is unset.")
else:
    # 1. 먼저 모든 instrument soft-delete. Manager / Connector cleanup에는
    #    ACTIVE instrument가 없어야 함. Manager 자체가 이미 삭제되었다면
    #    (예: 이전 cleanup 실행) 문제없이 건너뜀
    for label, connector_id, instrument_id in (
        ("CDP EVM", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_EVM),
        ("CDP SOLANA", CONNECTOR_ID_CDP, PAYMENT_INSTRUMENT_ID_CDP_SOL),
        ("Privy EVM", CONNECTOR_ID_PRIVY, PAYMENT_INSTRUMENT_ID_PRIVY_EVM),
        ("Privy SOLANA", CONNECTOR_ID_PRIVY, PAYMENT_INSTRUMENT_ID_PRIVY_SOL),
    ):
        if not instrument_id or not connector_id:
            continue
        try:
            payment_user_id = resolve_payment_user_id(instrument_id)
        except botocore.exceptions.ClientError as exc:
            code = exc.response.get("Error", {}).get("Code", "")
            msg = exc.response.get("Error", {}).get("Message", "")
            # Upstream Manager가 이전 실행에서 제거되었을 수 있음.
            # Service는 path에 따라 "AccessDenied: Payment manager not found" 또는
            # "ResourceNotFoundException"을 반환.
            # 두 경우 모두 이 row에서 정리할 항목 없음
            if code in ("AccessDeniedException", "ResourceNotFoundException") or "not found" in msg.lower():
                print(f"  ⚠️  Skipping {label} — already deleted ({code})")
                continue
            raise
        _safe_delete(
            dp_client_mgmt.delete_payment_instrument,
            f"Instrument {label} ({instrument_id})",
            paymentManagerArn=MANAGER_ARN,
            paymentConnectorId=connector_id,
            paymentInstrumentId=instrument_id,
            userId=payment_user_id,
        )

    # 2. 커넥터 → 관리자 → 자격 증명 공급자
    for label, connector_id in (
        ("CDP", CONNECTOR_ID_CDP),
        ("Privy", CONNECTOR_ID_PRIVY),
    ):
        if not connector_id:
            continue
        _safe_delete(
            cp_client.delete_payment_connector,
            f"{label} connector ({connector_id})",
            paymentManagerId=MANAGER_ID,
            paymentConnectorId=connector_id,
            clientToken=_client_token(),
        )

    _safe_delete(
        cp_client.delete_payment_manager,
        f"Manager {MANAGER_ID}",
        paymentManagerId=MANAGER_ID,
        clientToken=_client_token(),
    )

    for label, name in (
        ("CDP", CRED_PROVIDER_NAME_CDP),
        ("Privy", CRED_PROVIDER_NAME_PRIVY),
    ):
        if not name:
            continue
        _safe_delete(
            cred_client.delete_payment_credential_provider,
            f"{label} credential provider ({name})",
            name=name,
        )
    # 이후 Notebook 실행에서 방금 정리한 ID를 재사용하지 않도록
    # .env에서 삭제
    from utils import write_env_updates

    write_env_updates(
        {
            "MANAGER_ID": "",
            "MANAGER_ARN": "",
            "CRED_PROVIDER_NAME_CDP": "",
            "CREDENTIAL_PROVIDER_ARN_CDP": "",
            "CRED_PROVIDER_NAME_PRIVY": "",
            "CREDENTIAL_PROVIDER_ARN_PRIVY": "",
            "CONNECTOR_ID_CDP": "",
            "CONNECTOR_ID_PRIVY": "",
            "PAYMENT_INSTRUMENT_ID_CDP_EVM": "",
            "WALLET_ADDRESS_CDP_EVM": "",
            "PAYMENT_INSTRUMENT_ID_CDP_SOL": "",
            "WALLET_ADDRESS_CDP_SOL": "",
            "PAYMENT_INSTRUMENT_ID_PRIVY_EVM": "",
            "WALLET_ADDRESS_PRIVY_EVM": "",
            "PAYMENT_INSTRUMENT_ID_PRIVY_SOL": "",
            "WALLET_ADDRESS_PRIVY_SOL": "",
            "SESSION_ID_CDP": "",
            "SESSION_ID_PRIVY": "",
        }
    )
    print("\n✅ AgentCore Payments resources cleaned up.")
    print("💾 .env cleared of Manager/Connector/Instrument/Session IDs.")

### Local Build Artifact 제거

`cdk deploy`는 local artifact를 남깁니다. 각 CDK app에서 자체 생성하는 Python
environment인 `.venv/`, synthesized template인 `cdk.out/`, `__pycache__/`,
`outputs.json`입니다. Seller Lambda는 `npm install`을 통해 큰
`seller/lambda/node_modules/`(x402 facilitator dependency)를
남깁니다. Privy Wallet Hub frontend(§4.5e)는 runtime에 clone되므로
전체 `privy-delegation/` folder를 제거합니다. 다음 Notebook 실행 시
다시 clone됩니다. 이 항목에는 cloud state가 없으므로 안전하게 제거할 수 있으며
다음 실행을 깨끗한 상태로
시작할 수 있습니다.


In [ ]:
import pathlib
import shutil

ROOT = pathlib.Path(".").resolve()

# Directory는 seller/cdk/ 및 agent/cdk/ 아래에 있음(일부 helper import에서 생성된
# __pycache__ directory 포함). Source tree를 건드리지 않도록 glob 대신
# 정확한 path를 target으로 지정
targets = [
    ROOT / "seller" / "cdk" / ".venv",
    ROOT / "seller" / "cdk" / "cdk.out",
    ROOT / "seller" / "cdk" / "__pycache__",
    ROOT / "seller" / "cdk" / "outputs.json",
    # Seller Lambda — npm install이 x402 facilitator dependency를
    # 가져옴(약 16k file). Gitignore 대상이지만 local disk 공간 사용
    ROOT / "seller" / "lambda" / "node_modules",
    ROOT / "seller" / "lambda" / "package-lock.json",
    ROOT / "agent" / "cdk" / ".venv",
    ROOT / "agent" / "cdk" / "cdk.out",
    ROOT / "agent" / "cdk" / "__pycache__",
    ROOT / "agent" / "cdk" / "outputs.json",
    ROOT / "__pycache__",
    ROOT / "test" / "integration" / "__pycache__",
    # Privy Wallet Hub frontend(§4.5e). Clone 셀은 Notebook 실행마다
    # privy-io/aws-agentcore-sdk의 새 copy를 가져오므로 전체 folder를
    # 삭제해도 안전. 내부 항목은 vendored upstream tree 또는
    # runtime build 출력
    ROOT / "privy-delegation",
]

for path in targets:
    if not path.exists():
        print(f"  ↷ skip (absent): {path.relative_to(ROOT)}")
        continue
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()
    print(f"  🗑️  removed: {path.relative_to(ROOT)}")

print("\n✅ Local CDK artifacts removed.")

## 다음 단계

Public AgentCore Payments documentation에서 더 자세히 알아보세요.

- [AgentCore Payments overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments.html)
- [How it works](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-how-it-works.html)
- [Core concepts](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-concepts.html)
- [Process a payment](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-process-payment.html) — plugin reference, interrupt contract, network preferences
- [Connect to Bazaar](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-connect-bazaar.html) — make the seller discoverable through the AgentCore Registry
- [Agents that transact (announcement blog)](https://aws.amazon.com/blogs/machine-learning/agents-that-transact-introducing-amazon-bedrock-agentcore-payments-built-with-coinbase-and-stripe/)


# 축하합니다!

**Amazon Bedrock AgentCore Payments**를 사용하여 HTTP API의 metered access에
자동으로 결제하는 agent를 구축했습니다. 먼저 로컬 Strands agent로 실행한 다음
container로 package하여 **AgentCore Runtime**에 배포했습니다.

다룬 내용은 다음과 같습니다.

* **Self-contained 설정** — 기존 infrastructure 없이 Notebook에서 전체 AgentCore Payments
  stack(Credential Provider → Manager → Connector → Instrument → Session)을
  inline으로 provision
* **CDK seller stack** — 호출당 $0.01를 부과하는 간단한 API Gateway +
  Lambda
* **IAM role 분리** — `ManagementRole`은 session을 생성하고
  `ProcessPaymentRole`은 payment에 sign하며 문서가 아니라 IAM `Deny`로
  적용
* **Plugin을 사용하는 로컬 agent** — Strands agent 하나, `http_request` tool 하나,
  `AgentCorePaymentsPlugin`이 402 → ProcessPayment → retry를
  자동 처리
* **Runtime의 동일한 agent** — 동일한 agent 코드를 FastAPI container로 래핑하고
  CDK(`agent/cdk/`)를 통해 배포. Notebook은 로컬에서 사용한 것과 동일한
  prompt로 배포된 runtime 호출
* **Vendor-rooted identity** — 모든 data-plane operation이 Create 시 service에서 할당한
  CDP UUID 또는 Privy DID인 `paymentInstrument.userId`로 실행.
  Tenant mapping 및 DynamoDB 불필요
* **Budget 적용** — agent 실행 전에 operator가 `maxSpendAmount`
  설정
* **지출 검증** — `GetPaymentSession`으로 agent의 정확한 지출
  확인합니다.

**이 사용 사례의 확장 아이디어:**

* UI에서 token이 도착하는 즉시 표시하도록 Runtime의 streaming response 추가
  (`InvokeAgentRuntime`은 chunked output 지원)
* Agent가 이전 topic을 기억하도록 `AgentCoreMemorySessionManager`를 통해
  conversation memory 연결
* Static Fun Facts data를 Bedrock summarizer, third-party feed, AgentCore Registry lookup 등
  실제 upstream으로 교체
